In [ ]:
# === Google Colab 自動環境セットアップ ===
# ※ローカル環境では無視され、Colab環境でのみ自動でモジュールをインストールします
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/my_PRML'):
        print("リポジトリをダウンロード中...")
        !git clone https://github.com/miyayudai/my_PRML.git > /dev/null 2>&1
    
    print("必要なモジュールをインストール中...")
    %cd /content/my_PRML
    !pip install -q -r requirements.txt
    !pip install -q -e .
    
    print("作業ディレクトリをセットアップ中...")
    %cd /content/my_PRML/10
    print("準備完了！このまま下のセルを実行できます。")

# 第10章 近似推論 (Approximate Inference) — 演習問題完全解答・数値検証

本ノートブックは、C.M. Bishop 著『Pattern Recognition and Machine Learning (PRML)』**第10章「近似推論 (Approximate Inference)」**に収録されている **全39問（Exercises 10.1 〜 10.39）** の完全な論理証明、数理的思考プロセス、穴埋め形式のステップ導出、および自己完結した Python 数値検証コードを網羅した演習ノートブックです。

---

## 目次 (Table of Contents)

1. **変分推論の基礎、二変量ガウス因数分解、αダイバージェンス (Exercises 10.1 〜 10.9)**
   - [Exercise 10.1: 観測データの対数周辺尤度分解 (式 10.2 - 10.4)](#Exercise-10.1)
   - [Exercise 10.2: 二変量ガウス分布の因数分解変分近似 (式 10.13 - 10.15)](#Exercise-10.2)
   - [Exercise 10.3: KL(p || q) のラグランジュ乗数法による最小化 (式 10.16 - 10.17)](#Exercise-10.3)
   - [Exercise 10.4: ガウス分布による KL(p || q) 最小化 (モーメント整合)](#Exercise-10.4)
   - [Exercise 10.5: デルタ点推定近似変分法と EM アルゴリズムの等価性](#Exercise-10.5)
   - [Exercise 10.6: $\alpha$ ダイバージェンスの極限と KL ダイバージェンス (式 10.19)](#Exercise-10.6)
   - [Exercise 10.7: 1変量ガウス分布の因数分解変分更新式 (式 10.26 - 10.30)](#Exercise-10.7)
   - [Exercise 10.8: データ数大極限 $N \to \infty$ における変分精度事後分布の漸近挙動](#Exercise-10.8)
   - [Exercise 10.9: 期待精度の逆数に関する不動点方程式 (式 10.33)](#Exercise-10.9)

2. **変分モデル比較、変分混合ガウスモデル (V-GMM) (Exercises 10.10 〜 10.19)**
   - [Exercise 10.10: モデル事後分布の変分推論分解式 (10.34) の導出](#Exercise-10.10)
   - [Exercise 10.11: 変分下界最大化による最適モデル事後分布 $q^*(m) \propto p(m)\exp(\mathcal{L}_m)$ (10.36) の導出](#Exercise-10.11)
   - [Exercise 10.12: ベイズ混合ガウスにおける最適潜在変数事後分布 $q^*(\mathbf{Z})$ (10.48) の導出](#Exercise-10.12)
   - [Exercise 10.13: ガウス-ウィシャート変分事後分布 $q^*(\boldsymbol{\mu}_k, \mathbf{\Lambda}_k)$ (10.59) およびパラメータ更新式 (10.60-10.63) の導出](#Exercise-10.13)
   - [Exercise 10.14: ガウス-ウィシャート分布下の二次形式期待値公式 (10.64) の証明](#Exercise-10.14)
   - [Exercise 10.15: ディリクレ分布における混合比の期待値公式 (10.69) の証明](#Exercise-10.15)
   - [Exercise 10.16: ベイズ GMM 変分下界のデータ期待尤度項 (10.71) および潜在変数項 (10.72) の検証](#Exercise-10.16)
   - [Exercise 10.17: ベイズ GMM 変分下界の事前分布項およびエントロピー項 (10.73-10.77) の検証](#Exercise-10.17)
   - [Exercise 10.18: 変分下界 $\mathcal{L}$ の直接微分による GMM 再推定方程式の導出](#Exercise-10.18)
   - [Exercise 10.19: ベイズ混合ガウスの予測分布 $p(\widehat{\mathbf{x}}|\mathbf{X})$ が Student の t 分布混合となることの証明 (10.81)](#Exercise-10.19)

3. **大標本極限、対称性、特異点、ベイズ線形回帰変分推論、共役指数型分布族 (Exercises 10.20 〜 10.28)**
   - [Exercise 10.20: 大標本極限 $N \to \infty$ における変分ベイズ GMM の最尤推定 EM 解への漸近一致証明](#Exercise-10.20)
   - [Exercise 10.21: 混合モデルにおける置換対称性と $K!$ 個の同値モードの証明](#Exercise-10.21)
   - [Exercise 10.22: 単一モード変分近似と $K!$ 全対称化近似における事後予測分布の等価性](#Exercise-10.22)
   - [Exercise 10.23: 混合比 $\pi_k$ の点推定（事前分布なし）における変分下界最大化と最尤解一致](#Exercise-10.23)
   - [Exercise 10.24: 最大事後確率 (MAP) 推定における特異点再発と変分ベイズにおける特異点解消の比較論述](#Exercise-10.24)
   - [Exercise 10.25: 因数分解変分近似によるパラメータ共分散欠落と事後不確実性の過小評価](#Exercise-10.25)
   - [Exercise 10.26: 未知ノイズ精度 $\beta$ を含むベイズ線形回帰の変分推論更新式と下界の導出](#Exercise-10.26)
   - [Exercise 10.27: ベイズ線形回帰変分下界の各項 (10.107-10.112) の導出](#Exercise-10.27)
   - [Exercise 10.28: 共役指数型分布族の変分メッセージ伝播 (VMP) による GMM 更新式の統一的導出](#Exercise-10.28)

4. **局所変分法、Jaakkola-Jordan 境界、変分ロジスティック回帰、ADF & 期待伝播法 (EP) (Exercises 10.29 〜 10.39)**
   - [Exercise 10.29: 対数関数の凹性とルジャンドル変換による双対関数・元関数復元の証明](#Exercise-10.29)
   - [Exercise 10.30: 対数ロジスティック関数の凹性とテイラー展開による変分上界 (10.137) の導出](#Exercise-10.30)
   - [Exercise 10.31: Jaakkola-Jordan 下界関数 $\lambda(\xi)$ および二次形式下界 (10.144) の厳密な導出](#Exercise-10.31)
   - [Exercise 10.32: ベイズロジスティック回帰の逐次オンライン学習におけるガウス事後分布の閉包性](#Exercise-10.32)
   - [Exercise 10.33: 変分期待値関数 $\mathcal{Q}(\boldsymbol{\xi}, \boldsymbol{\xi}^{\mathrm{old}})$ の停留条件による $\xi_n^2$ 更新式 (10.163) の導出](#Exercise-10.33)
   - [Exercise 10.34: 変分下界 $\mathcal{L}(\boldsymbol{\xi})$ の直接最大化による再推定式の一致](#Exercise-10.34)
   - [Exercise 10.35: 変分ロジスティック回帰周辺下界 $\mathcal{L}(\boldsymbol{\xi})$ (10.164) のガウス積分による厳密導出](#Exercise-10.35)
   - [Exercise 10.36: Assumed Density Filtering (ADF) におけるモデルエビデンス逐次更新則 (10.242) の証明](#Exercise-10.36)
   - [Exercise 10.37: EP における共役事前分布因子の不動点不変性の証明](#Exercise-10.37)
   - [Exercise 10.38: クラッター問題における EP キャビティ分布 (10.214-10.215) および規格化定数 $Z_n$ (10.216) の導出](#Exercise-10.38)
   - [Exercise 10.39: クラッター問題における EP 更新平均 $\mathbf{m}^{\mathrm{new}}$ および分散 $v^{\mathrm{new}}$ (10.217-10.218) の導出](#Exercise-10.39)

---


In [1]:
# 共通ライブラリのインポートとパス設定
import sys, os
sys.path.append(os.path.abspath('../'))
import numpy as np
import scipy
from scipy import integrate
from scipy.optimize import minimize
from scipy.stats import multivariate_normal, wishart, gamma as gamma_dist
from scipy.special import psi, gamma, gammaln
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print("Environment successfully initialized for Chapter 10 exercises.")

Environment successfully initialized for Chapter 10 exercises.


### Exercise 10.1: 観測データの対数周辺尤度分解 (PRML 式 10.2 - 10.4)

#### 問題設定
観測データ $\mathbf{X}$ と潜在変数集合 $\mathbf{Z}$ に対し、任意の分布 $q(\mathbf{Z})$ を用いて、対数周辺分布 $\ln p(\mathbf{X})$ が以下のように厳密に2項に分解できることを示せ：
$$ \ln p(\mathbf{X}) = \mathcal{L}(q) + \mathrm{KL}(q \,||\, p) \tag{10.2} $$
ここで、変分下界 $\mathcal{L}(q)$ およびカルバック・ライブラー（KL）ダイバージェンス $\mathrm{KL}(q \,||\, p)$ は次式で定義される：
$$ \mathcal{L}(q) = \int q(\mathbf{Z}) \ln \left\{ \frac{p(\mathbf{X}, \mathbf{Z})}{q(\mathbf{Z})} \right\} d\mathbf{Z} \tag{10.3} $$
$$ \mathrm{KL}(q \,||\, p) = - \int q(\mathbf{Z}) \ln \left\{ \frac{p(\mathbf{Z}|\mathbf{X})}{q(\mathbf{Z})} \right\} d\mathbf{Z} = \int q(\mathbf{Z}) \ln \left\{ \frac{q(\mathbf{Z})}{p(\mathbf{Z}|\mathbf{X})} \right\} d\mathbf{Z} \tag{10.4} $$

#### 数理的導出と証明
1. 恒等式 $p(\mathbf{X}, \mathbf{Z}) = p(\mathbf{Z}|\mathbf{X}) p(\mathbf{X})$ より、両辺の対数を取ると：
   $$ \ln p(\mathbf{X}) = \ln p(\mathbf{X}, \mathbf{Z}) - \ln p(\mathbf{Z}|\mathbf{X}) $$
2. 任意確率密度関数 $q(\mathbf{Z})$ は規格化条件 $\int q(\mathbf{Z}) d\mathbf{Z} = 1$ を満たすため、両辺に $q(\mathbf{Z})$ を乗じて $\mathbf{Z}$ 全域で積分する：
   $$ \int q(\mathbf{Z}) \ln p(\mathbf{X}) d\mathbf{Z} = \ln p(\mathbf{X}) \int q(\mathbf{Z}) d\mathbf{Z} = \ln p(\mathbf{X}) $$
3. 右辺の積分において、項 $\ln q(\mathbf{Z})$ を足して引く（変形）：
   $$ \ln p(\mathbf{X}) = \int q(\mathbf{Z}) \left[ \ln p(\mathbf{X}, \mathbf{Z}) - \ln q(\mathbf{Z}) + \ln q(\mathbf{Z}) - \ln p(\mathbf{Z}|\mathbf{X}) \right] d\mathbf{Z} $$
   $$ = \int q(\mathbf{Z}) \ln \frac{p(\mathbf{X}, \mathbf{Z})}{q(\mathbf{Z})} d\mathbf{Z} + \int q(\mathbf{Z}) \ln \frac{q(\mathbf{Z})}{p(\mathbf{Z}|\mathbf{X})} d\mathbf{Z} $$
4. 第1項がまさに下界 $\mathcal{L}(q)$、第2項が事後分布 $p(\mathbf{Z}|\mathbf{X})$ に対する $q(\mathbf{Z})$ の KL ダイバージェンス $\mathrm{KL}(q \,||\, p)$ である。
   $\mathrm{KL}(q \,||\, p) \ge 0$ であるため、常に $\mathcal{L}(q) \le \ln p(\mathbf{X})$ が成り立ち、等号成立は $q(\mathbf{Z}) = p(\mathbf{Z}|\mathbf{X})$ のときに限られる。


In [2]:
# Exercise 10.1 数値検証: ln p(X) = L(q) + KL(q || p) の成立確認
import numpy as np

np.random.seed(42)
# 離散潜在変数 Z in {0, 1, 2} と観測 X の結合分布 p(X=x_obs, Z=k)
p_joint = np.array([0.15, 0.45, 0.20]) # p(x_obs, z)
p_X = np.sum(p_joint)
p_post = p_joint / p_X # p(z | x_obs)

# 任意のテスト分布 q(z)
q_test = np.array([0.3, 0.5, 0.2])

L_q = np.sum(q_test * np.log(p_joint / q_test))
KL_qp = np.sum(q_test * np.log(q_test / p_post))

ln_p_X = np.log(p_X)
sum_L_KL = L_q + KL_qp

print(f"ln p(X):   {ln_p_X:.8f}")
print(f"L(q):      {L_q:.8f}")
print(f"KL(q||p):  {KL_qp:.8f}")
print(f"L(q) + KL: {sum_L_KL:.8f}")

assert np.isclose(ln_p_X, sum_L_KL, atol=1e-12)
assert L_q <= ln_p_X + 1e-12
print("Exercise 10.1 verified: Decomposition ln p(X) = L(q) + KL holds exactly, L(q) <= ln p(X).")


ln p(X):   -0.22314355
L(q):      -0.26062441
KL(q||p):  0.03748086
L(q) + KL: -0.22314355
Exercise 10.1 verified: Decomposition ln p(X) = L(q) + KL holds exactly, L(q) <= ln p(X).


### Exercise 10.2 ⭐: 二変量ガウス分布の因数分解変分近似 (PRML 式 10.13 - 10.15)

#### 問題設定
2変数ガウス分布 $p(\mathbf{z}) = \mathcal{N}(\mathbf{z}|\boldsymbol{\mu}, \mathbf{\Lambda}^{-1})$ に対し、因数分解近似 $q(\mathbf{z}) = q_1(z_1)q_2(z_2)$ を適用する。
連立方程式 (10.13) および (10.15) を $\mathbb{E}[z_1] = m_1, \mathbb{E}[z_2] = m_2$ を用いて解き、
元の分布 $p(\mathbf{z})$ が正則（$\mathbf{\Lambda}$ が正定値）である限り、平均パラメータの一意解が $\mathbb{E}[z_1] = \mu_1, \mathbb{E}[z_2] = \mu_2$ となることを示せ。

#### 数理的導出と証明
1. 結合対数確率密度は以下のように展開される：
   $$ \ln p(\mathbf{z}) = -\frac{1}{2}\left[ \Lambda_{11}(z_1 - \mu_1)^2 + 2\Lambda_{12}(z_1 - \mu_1)(z_2 - \mu_2) + \Lambda_{22}(z_2 - \mu_2)^2 \right] + \mathrm{const} $$
2. 平均場近似の一般解 $\ln q_1^*(z_1) = \mathbb{E}_{z_2}[\ln p(\mathbf{z})] + \mathrm{const}$ より、
   $$ \ln q_1^*(z_1) = -\frac{1}{2}\Lambda_{11} z_1^2 + z_1 \left( \Lambda_{11}\mu_1 - \Lambda_{12}(m_2 - \mu_2) \right) + \mathrm{const} $$
   これにより、$q_1^*(z_1)$ はガウス分布 $\mathcal{N}(z_1 | m_1, \Lambda_{11}^{-1})$ となり、平均 $m_1$ は次式を満たす (10.13)：
   $$ m_1 = \mu_1 - \frac{\Lambda_{12}}{\Lambda_{11}}(m_2 - \mu_2) \iff \Lambda_{11}(m_1 - \mu_1) + \Lambda_{12}(m_2 - \mu_2) = 0 $$
3. 同様に、$q_2^*(z_2)$ について解くと (10.15)：
   $$ m_2 = \mu_2 - \frac{\Lambda_{21}}{\Lambda_{22}}(m_1 - \mu_1) \iff \Lambda_{21}(m_1 - \mu_1) + \Lambda_{22}(m_2 - \mu_2) = 0 $$
4. これを行列形式で表すと：
   $$ \begin{pmatrix} \Lambda_{11} & \Lambda_{12} \\ \Lambda_{21} & \Lambda_{22} \end{pmatrix} \begin{pmatrix} m_1 - \mu_1 \\ m_2 - \mu_2 \end{pmatrix} = \mathbf{\Lambda} (\mathbf{m} - \boldsymbol{\mu}) = \mathbf{0} $$
5. $p(\mathbf{z})$ が非特異（正則）ならば精度行列 $\mathbf{\Lambda}$ は正定値かつ可逆（$\det \mathbf{\Lambda} > 0$）であるため、自明解 $\mathbf{m} - \boldsymbol{\mu} = \mathbf{0}$、すなわち $m_1 = \mu_1, m_2 = \mu_2$ が唯一の解となる。
6. なお、変分事後分散は $\sigma_1^2 = 1 / \Lambda_{11}, \sigma_2^2 = 1 / \Lambda_{22}$ となり、真の周辺分散 $\Sigma_{11} = \frac{\Lambda_{22}}{\det \mathbf{\Lambda}} = \frac{1}{\Lambda_{11}(1 - \rho^2)} \ge \sigma_1^2$ より、常に真の分散を過小評価する。


In [3]:
# Exercise 10.2 数値検証: 二変量ガウス因数分解変分解
mu_true = np.array([2.0, -1.0])
cov_true = np.array([[2.0, 0.9], [0.9, 1.5]]) # 相関あり
Lambda_true = np.linalg.inv(cov_true)

# 連立一次方程式 Lambda @ (m - mu) = 0 の解
diff = np.linalg.solve(Lambda_true, np.zeros(2))
m_opt = mu_true + diff

# 最適分散
var_opt = 1.0 / np.diag(Lambda_true)
var_true = np.diag(cov_true)

print(f"True Mean: {mu_true} | Variational Mean: {m_opt}")
print(f"True Var:  {var_true} | Variational Var:  {var_opt}")

assert np.allclose(m_opt, mu_true, atol=1e-12)
assert np.all(var_opt < var_true) # 厳密な分散過小評価
print("Exercise 10.2 verified: Unique mean is mu, and variational variance strictly underestimates true variance.")


True Mean: [ 2. -1.] | Variational Mean: [ 2. -1.]
True Var:  [2.  1.5] | Variational Var:  [1.46  1.095]
Exercise 10.2 verified: Unique mean is mu, and variational variance strictly underestimates true variance.


### Exercise 10.3: KL(p || q) のラグランジュ乗数法による最小化 (PRML 式 10.16 - 10.17)

#### 問題設定
因数分解された変分分布 $q(\mathbf{Z}) = \prod_{i=1}^M q_i(\mathbf{Z}_i)$ に対し、逆向き KL ダイバージェンス $\mathrm{KL}(p \,||\, q)$ を最小化する問題を考える。
規格化条件 $\int q_i(\mathbf{Z}_i) d\mathbf{Z}_i = 1$ をラグランジュ乗数法で導入し、他の因子を固定したときの最適解が次式で与えられることを示せ：
$$ q_i^*(\mathbf{Z}_i) = \int p(\mathbf{Z}) \prod_{j \ne i} d\mathbf{Z}_j = p(\mathbf{Z}_i) \tag{10.17} $$

#### 数理的導出と証明
1. 目的関数 $\mathrm{KL}(p \,||\, q)$ を展開する：
   $$ \mathrm{KL}(p \,||\, q) = \int p(\mathbf{Z}) \ln \frac{p(\mathbf{Z})}{q(\mathbf{Z})} d\mathbf{Z} = -\int p(\mathbf{Z}) \sum_{j=1}^M \ln q_j(\mathbf{Z}_j) d\mathbf{Z} + \mathrm{const} $$
2. 特定の因子 $q_i(\mathbf{Z}_i)$ に依存する項を取り出す：
   $$ -\int p(\mathbf{Z}) \ln q_i(\mathbf{Z}_i) d\mathbf{Z} = -\int \left( \int p(\mathbf{Z}) \prod_{j \ne i} d\mathbf{Z}_j \right) \ln q_i(\mathbf{Z}_i) d\mathbf{Z}_i = -\int p(\mathbf{Z}_i) \ln q_i(\mathbf{Z}_i) d\mathbf{Z}_i $$
3. 規格化拘束条件 $\int q_i(\mathbf{Z}_i) d\mathbf{Z}_i - 1 = 0$ に対するラグランジュ乗数を $\lambda$ としてラグランジアンを構成する：
   $$ \widetilde{L}(q_i, \lambda) = -\int p(\mathbf{Z}_i) \ln q_i(\mathbf{Z}_i) d\mathbf{Z}_i + \lambda \left( \int q_i(\mathbf{Z}_i) d\mathbf{Z}_i - 1 \right) $$
4. $q_i(\mathbf{Z}_i)$ に関する汎関数微分を計算して 0 とおく：
   $$ \frac{\delta \widetilde{L}}{\delta q_i(\mathbf{Z}_i)} = -\frac{p(\mathbf{Z}_i)}{q_i(\mathbf{Z}_i)} + \lambda = 0 \implies q_i(\mathbf{Z}_i) = \frac{p(\mathbf{Z}_i)}{\lambda} $$
5. 規格化条件 $\int q_i(\mathbf{Z}_i) d\mathbf{Z}_i = 1$ より $\lambda = \int p(\mathbf{Z}_i) d\mathbf{Z}_i = 1$ となり、
   $$ q_i^*(\mathbf{Z}_i) = p(\mathbf{Z}_i) $$
   が導かれる。すなわち、$\mathrm{KL}(p \,||\, q)$ の最小化は、各因子の真の周辺分布への一致（モーメント整合）に対応する。


In [4]:
# Exercise 10.3 数値検証: KL(p || q) 最小化における q_i(Z_i) = p(Z_i)
# 2次元離散結合分布 p(z1, z2)
P = np.array([[0.1, 0.2, 0.1],
              [0.05, 0.35, 0.2]])
p1 = np.sum(P, axis=1) # p(z1)
p2 = np.sum(P, axis=0) # p(z2)

# 理論解 q*(z1, z2) = p(z1) * p(z2)
Q_opt = np.outer(p1, p2)
kl_opt = np.sum(P * np.log(P / Q_opt))

# ランダムな摂動分布との比較
for _ in range(5):
    q1_pert = p1 + np.random.randn(len(p1)) * 0.05
    q1_pert = np.clip(q1_pert, 0.01, None); q1_pert /= np.sum(q1_pert)
    Q_pert = np.outer(q1_pert, p2)
    kl_pert = np.sum(P * np.log(P / Q_pert))
    assert kl_pert > kl_opt

print(f"Optimal KL(p || q): {kl_opt:.6f}")
print("Exercise 10.3 verified: q_i(Z_i) = p(Z_i) strictly minimizes KL(p || q).")


Optimal KL(p || q): 0.026065
Exercise 10.3 verified: q_i(Z_i) = p(Z_i) strictly minimizes KL(p || q).


### Exercise 10.4: ガウス分布による KL(p || q) 最小化 (モーメント整合)

#### 問題設定
任意の固定された分布 $p(\mathbf{x})$ に対し、ガウス分布 $q(\mathbf{x}) = \mathcal{N}(\mathbf{x}|\boldsymbol{\mu}, \mathbf{\Sigma})$ を用いて近似する。
$\mathrm{KL}(p \,||\, q)$ を $\boldsymbol{\mu}$ および $\mathbf{\Sigma}$ について最小化すると、
$\boldsymbol{\mu} = \mathbb{E}_p[\mathbf{x}]$、$\mathbf{\Sigma} = \mathrm{cov}_p[\mathbf{x}]$ となることを示せ。

#### 数理的導出と証明
1. $\mathrm{KL}(p \,||\, q)$ の定義式：
   $$ \mathrm{KL}(p \,||\, q) = \int p(\mathbf{x}) \left[ \ln p(\mathbf{x}) - \ln q(\mathbf{x}) \right] d\mathbf{x} = -\int p(\mathbf{x}) \ln q(\mathbf{x}) d\mathbf{x} + \mathrm{const} $$
2. ガウス分布 $q(\mathbf{x})$ の対数密度：
   $$ \ln q(\mathbf{x}) = -\frac{D}{2}\ln(2\pi) - \frac{1}{2}\ln|\mathbf{\Sigma}| - \frac{1}{2}(\mathbf{x} - \boldsymbol{\mu})^{\mathrm{T}}\mathbf{\Sigma}^{-1}(\mathbf{x} - \boldsymbol{\mu}) $$
3. $p(\mathbf{x})$ の下での期待値：
   $$ \mathbb{E}_p[-\ln q(\mathbf{x})] = \frac{1}{2}\ln|\mathbf{\Sigma}| + \frac{1}{2}\mathbb{E}_p\left[ \mathrm{Tr}\left( \mathbf{\Sigma}^{-1}(\mathbf{x} - \boldsymbol{\mu})(\mathbf{x} - \boldsymbol{\mu})^{\mathrm{T}} \right) \right] + \mathrm{const} $$
   $$ = \frac{1}{2}\ln|\mathbf{\Sigma}| + \frac{1}{2}\mathrm{Tr}\left( \mathbf{\Sigma}^{-1} \left( \mathrm{cov}_p[\mathbf{x}] + (\mathbb{E}_p[\mathbf{x}] - \boldsymbol{\mu})(\mathbb{E}_p[\mathbf{x}] - \boldsymbol{\mu})^{\mathrm{T}} \right) \right) + \mathrm{const} $$
4. $\boldsymbol{\mu}$ に関する微分：
   $$ \nabla_{\boldsymbol{\mu}}\mathrm{KL}(p \,||\, q) = -\mathbf{\Sigma}^{-1}(\mathbb{E}_p[\mathbf{x}] - \boldsymbol{\mu}) = \mathbf{0} \implies \boldsymbol{\mu}^* = \mathbb{E}_p[\mathbf{x}] $$
5. 最適 $\boldsymbol{\mu}^*$ のもとで、$\mathbf{\Sigma}$ (または精度行列 $\mathbf{\Lambda} = \mathbf{\Sigma}^{-1}$) に関する微分：
   $$ \frac{\partial}{\partial \mathbf{\Sigma}^{-1}}\mathrm{KL}(p \,||\, q) = -\frac{1}{2}\mathbf{\Sigma} + \frac{1}{2}\mathrm{cov}_p[\mathbf{x}] = \mathbf{0} \implies \mathbf{\Sigma}^* = \mathrm{cov}_p[\mathbf{x}] $$
   したがって、$\mathrm{KL}(p \,||\, q)$ 最小化は、真の分布 $p(\mathbf{x})$ の平均および共分散行列と完全に一致する。


In [5]:
# Exercise 10.4 数値検証: KL(p || q) 最小化ガウスパラメータ
# 非ガウス分布 p(x): 2峰性ガウス混合分布
weights_true = np.array([0.4, 0.6])
means_true = np.array([[-2.0, -1.0], [2.0, 1.0]])
covs_true = np.array([[[0.5, 0.1], [0.1, 0.4]],
                      [[0.8, -0.2], [-0.2, 0.6]]])

# 真の期待値と共分散
E_x_true = np.sum(weights_true[:, None] * means_true, axis=0)
cov_x_true = np.zeros((2, 2))
for k in range(2):
    diff = means_true[k] - E_x_true
    cov_x_true += weights_true[k] * (covs_true[k] + np.outer(diff, diff))

# サンプリングによる近似評価
samples = []
for _ in range(50000):
    k = np.random.choice(2, p=weights_true)
    samples.append(np.random.multivariate_normal(means_true[k], covs_true[k]))
samples = np.array(samples)

sample_mean = np.mean(samples, axis=0)
sample_cov = np.cov(samples, rowvar=False)

print("Theory Mean:", E_x_true, "| Sample Mean:", np.round(sample_mean, 4))
print("Theory Cov:\n", cov_x_true, "\nSample Cov:\n", np.round(sample_cov, 4))

assert np.allclose(E_x_true, sample_mean, atol=0.03)
assert np.allclose(cov_x_true, sample_cov, atol=0.03)
print("Exercise 10.4 verified: Optimal Gaussian parameters match true 1st and 2nd moments.")


Theory Mean: [0.4 0.2] | Sample Mean: [0.403  0.2031]
Theory Cov:
 [[4.52 1.84]
 [1.84 1.48]] 
Sample Cov:
 [[4.5128 1.8372]
 [1.8372 1.4839]]
Exercise 10.4 verified: Optimal Gaussian parameters match true 1st and 2nd moments.


### Exercise 10.5: 点推定近似と EM アルゴリズムの等価性

#### 問題設定
確率変数全体 $\mathbf{Z}$ が潜在変数 $\mathbf{z}$ とモデルパラメータ $\boldsymbol{\theta}$ から成るとし、因数分解変分分布 $q(\mathbf{z}, \boldsymbol{\theta}) = q_z(\mathbf{z}) q_\theta(\boldsymbol{\theta})$ を用いる。
ここで、$q_\theta(\boldsymbol{\theta})$ としてデルタ関数による点推定近似 $q_\theta(\boldsymbol{\theta}) = \delta(\boldsymbol{\theta} - \boldsymbol{\theta}_0)$ を採用する。
このとき、因数分解変分最適化が EM アルゴリズム（Eステップで $q_z(\mathbf{z})$ を最適化し、Mステップで $\boldsymbol{\theta}_0$ に関して期待完全対数事後確率を最大化する）と厳密に等価であることを示せ。

#### 数理的導出と証明
1. 変分下界 $\mathcal{L}(q_z, \boldsymbol{\theta}_0)$ を計算する：
   $$ \mathcal{L}(q_z, \boldsymbol{\theta}_0) = \iint q_z(\mathbf{z})\delta(\boldsymbol{\theta} - \boldsymbol{\theta}_0) \ln \frac{p(\mathbf{X}, \mathbf{z}, \boldsymbol{\theta})}{q_z(\mathbf{z})\delta(\boldsymbol{\theta} - \boldsymbol{\theta}_0)} d\mathbf{z} d\boldsymbol{\theta} $$
2. $\boldsymbol{\theta}$ に関する積分を実行すると、$\boldsymbol{\theta} = \boldsymbol{\theta}_0$ に固定され、
   $$ \mathcal{L}(q_z, \boldsymbol{\theta}_0) = \mathbb{E}_{q_z}[\ln p(\mathbf{X}, \mathbf{z}, \boldsymbol{\theta}_0)] - \mathbb{E}_{q_z}[\ln q_z(\mathbf{z})] + \mathrm{const} $$
3. **Eステップ**: $\boldsymbol{\theta}_0$ を固定して $q_z(\mathbf{z})$ を最大化する：
   $$ \ln q_z^*(\mathbf{z}) = \ln p(\mathbf{X}, \mathbf{z}, \boldsymbol{\theta}_0) + \mathrm{const} \implies q_z^*(\mathbf{z}) = p(\mathbf{z}|\mathbf{X}, \boldsymbol{\theta}_0) $$
   これは現在のパラメータ推定値 $\boldsymbol{\theta}_0$ のもとでの潜在変数の事後確率計算（通常のEMアルゴリズムのEステップ）そのものである。
4. **Mステップ**: 最適化された $q_z^*(\mathbf{z})$ を固定して $\boldsymbol{\theta}_0$ について最大化する：
   $$ \boldsymbol{\theta}_0^* = \arg\max_{\boldsymbol{\theta}_0} \mathcal{L}(q_z^*, \boldsymbol{\theta}_0) = \arg\max_{\boldsymbol{\theta}_0} \mathbb{E}_{q_z^*}[\ln p(\mathbf{X}, \mathbf{z}, \boldsymbol{\theta}_0)] $$
   事前分布 $p(\boldsymbol{\theta})$ を含める場合、$p(\mathbf{X}, \mathbf{z}, \boldsymbol{\theta}_0) = p(\mathbf{X}, \mathbf{z}|\boldsymbol{\theta}_0)p(\boldsymbol{\theta}_0)$ より MAP-EM の M ステップに一致し、無情報事前分布の場合は MLE-EM の M ステップに一致する。


In [6]:
# Exercise 10.5 数値検証: デルタ近似変分法と EM アルゴリズムの同一ステップ遷移
X_1d = np.array([-1.2, -0.8, -1.0, 1.5, 1.8, 2.1])
# 混合比 0.5 固定の 1次元 2成分 GMM (分散 1 固定、平均 mu1, mu2 の推定)
mu = np.array([-0.5, 0.5])

for it in range(5):
    # Eステップ: q(z_n) = p(z_n | x_n, mu)
    d1 = np.exp(-0.5 * (X_1d - mu[0])**2)
    d2 = np.exp(-0.5 * (X_1d - mu[1])**2)
    gamma1 = d1 / (d1 + d2)
    gamma2 = d2 / (d1 + d2)
    
    # Mステップ: L(q, mu) を mu について最大化
    mu_new = np.array([np.sum(gamma1 * X_1d) / np.sum(gamma1),
                       np.sum(gamma2 * X_1d) / np.sum(gamma2)])
    mu = mu_new

print("Final estimated centers via point-mass Variational EM:", np.round(mu, 4))
assert np.isclose(mu[0], -1.0, atol=0.1) and np.isclose(mu[1], 1.8, atol=0.1)
print("Exercise 10.5 verified: Variational inference with delta-factor recovers EM algorithm exactly.")


Final estimated centers via point-mass Variational EM: [-0.9245  1.7329]
Exercise 10.5 verified: Variational inference with delta-factor recovers EM algorithm exactly.


### Exercise 10.6: $\alpha$ ダイバージェンスの極限と KL ダイバージェンス (PRML 式 10.19)

#### 問題設定
$\alpha$ ダイバージェンス族は以下で定義される：
$$ \mathrm{D}_\alpha(p \,||\, q) = \frac{4}{1 - \alpha^2} \left( 1 - \int p(x)^{\frac{1+\alpha}{2}} q(x)^{\frac{1-\alpha}{2}} dx \right) \tag{10.19} $$
$\alpha \to 1$ の極限が $\mathrm{KL}(p \,||\, q)$ に、$\alpha \to -1$ の極限が $\mathrm{KL}(q \,||\, p)$ に収束することを、$p^\epsilon = 1 + \epsilon \ln p + \mathcal{O}(\epsilon^2)$ の展開を用いて示せ。

#### 数理的導出と証明
1. $\epsilon = \frac{1 - \alpha}{2}$ と定義すると、$\alpha \to 1 \iff \epsilon \to 0$ である。
   分母は $1 - \alpha^2 = (1 - \alpha)(1 + \alpha) = 2\epsilon(2 - 2\epsilon) = 4\epsilon(1 - \epsilon)$ と書ける。
2. 被積分関数について：
   $$ \frac{1+\alpha}{2} = 1 - \epsilon, \quad \frac{1-\alpha}{2} = \epsilon $$
   $$ p(x)^{1-\epsilon} q(x)^\epsilon = p(x) \left( \frac{q(x)}{p(x)} \right)^\epsilon = p(x) \exp\left( \epsilon \ln \frac{q(x)}{p(x)} \right) $$
3. テイラー展開 $\exp(u) = 1 + u + \mathcal{O}(u^2)$ を適用する：
   $$ p(x)^{1-\epsilon} q(x)^\epsilon = p(x) \left[ 1 + \epsilon \ln \frac{q(x)}{p(x)} + \mathcal{O}(\epsilon^2) \right] = p(x) - \epsilon p(x) \ln \frac{p(x)}{q(x)} + \mathcal{O}(\epsilon^2) $$
4. 積分を実行すると、$\int p(x) dx = 1$ より：
   $$ \int p(x)^{1-\epsilon} q(x)^\epsilon dx = 1 - \epsilon \int p(x) \ln \frac{p(x)}{q(x)} dx + \mathcal{O}(\epsilon^2) = 1 - \epsilon \mathrm{KL}(p \,||\, q) + \mathcal{O}(\epsilon^2) $$
5. これを $\mathrm{D}_\alpha(p \,||\, q)$ の式に代入する：
   $$ \lim_{\alpha \to 1} \mathrm{D}_\alpha(p \,||\, q) = \lim_{\epsilon \to 0} \frac{4}{4\epsilon(1 - \epsilon)} \left[ \epsilon \mathrm{KL}(p \,||\, q) + \mathcal{O}(\epsilon^2) \right] = \mathrm{KL}(p \,||\, q) $$
6. 同様に、$\alpha \to -1$ の極限では、$\delta = \frac{1+\alpha}{2} \to 0$ と置くことで $p$ と $q$ の役割が反転し、
   $$ \lim_{\alpha \to -1} \mathrm{D}_\alpha(p \,||\, q) = \mathrm{KL}(q \,||\, p) $$
   が得られる。


In [7]:
# Exercise 10.6 数値検証: alpha -> 1 および alpha -> -1 での D_alpha 収束性
p_vec = np.array([0.2, 0.5, 0.3])
q_vec = np.array([0.35, 0.40, 0.25])

kl_pq = np.sum(p_vec * np.log(p_vec / q_vec))
kl_qp = np.sum(q_vec * np.log(q_vec / p_vec))

def D_alpha(p, q, a):
    return (4.0 / (1.0 - a**2)) * (1.0 - np.sum(p**((1.0 + a)/2.0) * q**((1.0 - a)/2.0)))

alphas_forward = [0.8, 0.95, 0.99, 0.999, 0.9999]
for a in alphas_forward:
    val = D_alpha(p_vec, q_vec, a)
    print(f"alpha = {a:7.4f} -> D_alpha = {val:.8f} (True KL(p||q) = {kl_pq:.8f})")

assert np.isclose(D_alpha(p_vec, q_vec, 0.9999), kl_pq, rtol=1e-3)
assert np.isclose(D_alpha(p_vec, q_vec, -0.9999), kl_qp, rtol=1e-3)
print("Exercise 10.6 verified: D_alpha converges continuously to KL(p||q) as alpha->1 and KL(q||p) as alpha->-1.")


alpha =  0.8000 -> D_alpha = 0.05491436 (True KL(p||q) = 0.05434509)
alpha =  0.9500 -> D_alpha = 0.05448554 (True KL(p||q) = 0.05434509)
alpha =  0.9900 -> D_alpha = 0.05437308 (True KL(p||q) = 0.05434509)
alpha =  0.9990 -> D_alpha = 0.05434788 (True KL(p||q) = 0.05434509)
alpha =  0.9999 -> D_alpha = 0.05434536 (True KL(p||q) = 0.05434509)
Exercise 10.6 verified: D_alpha converges continuously to KL(p||q) as alpha->1 and KL(q||p) as alpha->-1.


### Exercise 10.7: 1変量ガウス分布の因数分解変分更新式 (PRML 式 10.26 - 10.30)

#### 問題設定
1変量ガウス分布の尤度関数 $p(\mathbf{x}|\mu, \tau) = \prod_{n=1}^N \mathcal{N}(x_n|\mu, \tau^{-1})$ に対し、
共役事前分布 $p(\mu|\tau) = \mathcal{N}(\mu|\mu_0, (\lambda_0 \tau)^{-1})$, $p(\tau) = \mathrm{Gam}(\tau|a_0, b_0)$ を置く。
因数分解近似 $q(\mu, \tau) = q_\mu(\mu)q_\tau(\tau)$ を用いるとき、
$q_\mu(\mu) = \mathcal{N}(\mu|\mu_N, \lambda_N^{-1})$（式 10.26, 10.27）および $q_\tau(\tau) = \mathrm{Gam}(\tau|a_N, b_N)$（式 10.29, 10.30）となることを証明せよ。

#### 数理的導出と証明
1. 結合対数確率は以下のように表される：
   $$ \ln p(\mathbf{x}, \mu, \tau) = \ln p(\tau) + \ln p(\mu|\tau) + \sum_{n=1}^N \ln p(x_n|\mu, \tau) $$
   $$ = (a_0 - 1)\ln \tau - b_0 \tau + \frac{1}{2}\ln \tau - \frac{\lambda_0 \tau}{2}(\mu - \mu_0)^2 + \frac{N}{2}\ln \tau - \frac{\tau}{2}\sum_{n=1}^N (x_n - \mu)^2 + \mathrm{const} $$
2. $q_\mu(\mu)$ の更新：$\tau$ に関する期待値を取ると、
   $$ \ln q_\mu(\mu) = -\frac{\mathbb{E}[\tau]}{2}\left[ \lambda_0(\mu - \mu_0)^2 + \sum_{n=1}^N (x_n - \mu)^2 \right] + \mathrm{const} $$
   $\mu$ の2次の項と1次の項をまとめると：
   $$ -\frac{\mathbb{E}[\tau]}{2} \left[ (\lambda_0 + N)\mu^2 - 2\mu (\lambda_0 \mu_0 + N \bar{x}) \right] $$
   したがって、$q_\mu(\mu)$ はガウス分布であり、
   $$ \mu_N = \frac{\lambda_0 \mu_0 + N \bar{x}}{\lambda_0 + N} \tag{10.26} $$
   $$ \lambda_N = (\lambda_0 + N)\mathbb{E}[\tau] \tag{10.27} $$
3. $q_\tau(\tau)$ の更新：$\mu$ に関する期待値を取ると、
   $$ \ln q_\tau(\tau) = \left( a_0 + \frac{N+1}{2} - 1 \right)\ln \tau - \tau \left[ b_0 + \frac{1}{2}\mathbb{E}_\mu\left( \lambda_0(\mu - \mu_0)^2 + \sum_{n=1}^N (x_n - \mu)^2 \right) \right] + \mathrm{const} $$
   これはガンマ分布 $\mathrm{Gam}(\tau|a_N, b_N)$ の対数密度に一致し、
   $$ a_N = a_0 + \frac{N+1}{2} \tag{10.29} $$
   $$ b_N = b_0 + \frac{1}{2}\mathbb{E}_\mu\left[ \lambda_0(\mu - \mu_0)^2 + \sum_{n=1}^N (x_n - \mu)^2 \right] \tag{10.30} $$


In [8]:
# Exercise 10.7 数値検証: 1変量ガウス変分パラメータ更新の収束
from common.variational_utils import variational_gaussian_1d

X_synth = np.array([0.5, 1.2, 0.8, 1.5, 2.0, 1.8, 0.9, 1.3])
N = len(X_synth)
x_bar = np.mean(X_synth)

mu_0, lambda_0, a_0, b_0 = 0.0, 1.0, 1.0, 1.0
history = variational_gaussian_1d(X_synth, mu_0, lambda_0, a_0, b_0, max_iter=20)
mu_N, lambda_N, a_N, b_N = history[-1]

# 理論式 10.26, 10.29
expected_mu_N = (lambda_0 * mu_0 + N * x_bar) / (lambda_0 + N)
expected_a_N = a_0 + (N + 1) / 2.0

print(f"mu_N: {mu_N:.6f} (Theory: {expected_mu_N:.6f})")
print(f"a_N:  {a_N:.6f} (Theory: {expected_a_N:.6f})")

assert np.isclose(mu_N, expected_mu_N, atol=1e-10)
assert np.isclose(a_N, expected_a_N, atol=1e-10)
print("Exercise 10.7 verified: q_mu and q_tau parameter updates strictly match theoretical derivations.")


mu_N: 1.111111 (Theory: 1.111111)
a_N:  5.500000 (Theory: 5.500000)
Exercise 10.7 verified: q_mu and q_tau parameter updates strictly match theoretical derivations.


### Exercise 10.8: データ数大極限 $N \to \infty$ における変分精度事後分布の漸近挙動

#### 問題設定
1変量ガウス分布の変分精度事後分布 $q_\tau(\tau) = \mathrm{Gam}(\tau|a_N, b_N)$ に対し、ガンマ分布の平均・分散の標準公式を用いて、
$N \to \infty$ の極限で平均 $\mathbb{E}[\tau] \to 1/\sigma_{\mathrm{ML}}^2$ となり、分散 $\mathrm{var}[\tau] \to 0$ となることを示せ。

#### 数理的導出と証明
1. 式 (10.29) より、$a_N = a_0 + \frac{N+1}{2} = \frac{N}{2} + \mathcal{O}(1)$ である。
2. 式 (10.30) において、大数の法則より標本平均 $\bar{x} \to \mu$、$\mu_N \to \bar{x}$、$\lambda_N \to \infty$ となる。
   したがって、$\mathbb{E}[(\mu - \mu_N)^2] = \lambda_N^{-1} \to 0$ となり、
   $$ b_N = b_0 + \frac{1}{2}\sum_{n=1}^N (x_n - \bar{x})^2 + \mathcal{O}(1) = \frac{N}{2}\sigma_{\mathrm{ML}}^2 + \mathcal{O}(1) $$
   ここで $\sigma_{\mathrm{ML}}^2 = \frac{1}{N}\sum_{n=1}^N (x_n - \bar{x})^2$ である。
3. ガンマ分布の期待値公式 $\mathbb{E}[\tau] = \frac{a_N}{b_N}$ より：
   $$ \lim_{N \to \infty} \mathbb{E}[\tau] = \lim_{N \to \infty} \frac{\frac{N}{2} + \mathcal{O}(1)}{\frac{N}{2}\sigma_{\mathrm{ML}}^2 + \mathcal{O}(1)} = \frac{1}{\sigma_{\mathrm{ML}}^2} $$
4. ガンマ分布の分散公式 $\mathrm{var}[\tau] = \frac{a_N}{b_N^2}$ より：
   $$ \lim_{N \to \infty} \mathrm{var}[\tau] = \lim_{N \to \infty} \frac{\frac{N}{2} + \mathcal{O}(1)}{\left(\frac{N}{2}\sigma_{\mathrm{ML}}^2\right)^2} = \lim_{N \to \infty} \frac{2}{N \sigma_{\mathrm{ML}}^4} = 0 $$
   したがって、$N \to \infty$ で変分事後分布は最尤推定解を中心とするデルタ関数へと退化する。


In [9]:
# Exercise 10.8 数値検証: N -> infty での E[tau] -> 1/sigma_ML^2 および var[tau] -> 0
true_mean = 3.0
true_var = 4.0
sample_sizes = [20, 200, 2000, 20000]

for N_val in sample_sizes:
    X_large = np.random.normal(loc=true_mean, scale=np.sqrt(true_var), size=N_val)
    sigma_ml_sq = np.var(X_large)
    hist = variational_gaussian_1d(X_large, max_iter=20)
    _, _, a_final, b_final = hist[-1]
    
    e_tau = a_final / b_final
    var_tau = a_final / (b_final**2)
    
    print(f"N = {N_val:5d}: E[tau] = {e_tau:.5f} (1/sigma_ML^2 = {1.0/sigma_ml_sq:.5f}), var[tau] = {var_tau:.2e}")

assert np.isclose(e_tau, 1.0 / sigma_ml_sq, rtol=0.01)
assert var_tau < 1e-4
print("Exercise 10.8 verified: E[tau] converges to 1/sigma_ML^2 and var[tau] vanishes as N -> infinity.")


N =    20: E[tau] = 0.34316 (1/sigma_ML^2 = 0.34316), var[tau] = 1.12e-02
N =   200: E[tau] = 0.25454 (1/sigma_ML^2 = 0.25454), var[tau] = 6.45e-04
N =  2000: E[tau] = 0.24591 (1/sigma_ML^2 = 0.24591), var[tau] = 6.04e-05
N = 20000: E[tau] = 0.25523 (1/sigma_ML^2 = 0.25523), var[tau] = 6.51e-06
Exercise 10.8 verified: E[tau] converges to 1/sigma_ML^2 and var[tau] vanishes as N -> infinity.


### Exercise 10.9: 期待精度の逆数に関する不動点方程式 (PRML 式 10.33)

#### 問題設定
1変量ガウス分布に対する因数分解変分推論において、無情報事前分布極限 $\mu_0 = \lambda_0 = a_0 = b_0 = 0$ を考える。
ガンマ分布の期待値公式 $\mathbb{E}[\tau] = a_N / b_N$、ならびに更新式 (10.26), (10.27), (10.29), (10.30) を用いて、
期待精度の逆数 $1/\mathbb{E}[\tau]$ が不偏分散推定量
$$ \frac{1}{\mathbb{E}[\tau]} = \frac{1}{N - 1} \sum_{n=1}^N (x_n - \bar{x})^2 \tag{10.33} $$
に厳密に一致することを導出せよ。

#### 数理的導出と証明
1. 無情報事前分布において、式 (10.29) より $a_N = \frac{N}{2}$ である。
2. 式 (10.30) より $b_N = \frac{1}{2} \sum_{n=1}^N \mathbb{E}_\mu [(x_n - \mu)^2]$ である。
3. 二乗項を展開すると：
   $$ \sum_{n=1}^N \mathbb{E}_\mu [(x_n - \mu)^2] = \sum_{n=1}^N \left( x_n^2 - 2 x_n \mathbb{E}[\mu] + \mathbb{E}[\mu^2] \right) $$
4. 式 (10.26) より $\mathbb{E}[\mu] = \mu_N = \bar{x}$、式 (10.27) より精度 $\lambda_N = N \mathbb{E}[\tau]$ であるため、
   $$ \mathbb{E}[\mu^2] = \mu_N^2 + \lambda_N^{-1} = \bar{x}^2 + \frac{1}{N \mathbb{E}[\tau]} $$
5. これらを代入して総和をとると：
   $$ \sum_{n=1}^N \mathbb{E}_\mu [(x_n - \mu)^2] = \sum_{n=1}^N (x_n - \bar{x})^2 + \frac{1}{\mathbb{E}[\tau]} $$
6. したがって $b_N = \frac{1}{2} \sum_{n=1}^N (x_n - \bar{x})^2 + \frac{1}{2 \mathbb{E}[\tau]}$ となる。
7. $\mathbb{E}[\tau] = \frac{a_N}{b_N} \iff \frac{1}{\mathbb{E}[\tau]} = \frac{b_N}{a_N}$ に代入すると：
   $$ \frac{1}{\mathbb{E}[\tau]} = \frac{\frac{1}{2} \sum_{n=1}^N (x_n - \bar{x})^2 + \frac{1}{2 \mathbb{E}[\tau]}}{\frac{N}{2}} = \frac{1}{N} \sum_{n=1}^N (x_n - \bar{x})^2 + \frac{1}{N \mathbb{E}[\tau]} $$
8. 両辺から $\frac{1}{N \mathbb{E}[\tau]}$ を引くと：
   $$ \left( 1 - \frac{1}{N} \right) \frac{1}{\mathbb{E}[\tau]} = \frac{N - 1}{N} \frac{1}{\mathbb{E}[\tau]} = \frac{1}{N} \sum_{n=1}^N (x_n - \bar{x})^2 $$
9. 両辺に $\frac{N}{N - 1}$ を乗じることで、式 (10.33) が厳密に得られる：
   $$ \frac{1}{\mathbb{E}[\tau]} = \frac{1}{N - 1} \sum_{n=1}^N (x_n - \bar{x})^2 $$


In [10]:
# Exercise 10.9 数値検証: 式 (10.33) の代数的整合性と不偏分散一致
X_pts = np.array([1.2, 2.3, 1.8, 2.7, 3.1])
N_pts = len(X_pts)
x_bar_pts = np.mean(X_pts)

# 式 (10.33) 理論値: 不偏分散 s^2
sample_var_unbiased = np.sum((X_pts - x_bar_pts)**2) / (N_pts - 1)

# 無情報事前分布 (mu_0 = lambda_0 = a_0 = b_0 = 0) における変分座標上昇反復
# a_N = N / 2, b_N = 0.5 * sum (x_n - x_bar)^2 + 0.5 / E[tau]
E_tau = 1.0
for it in range(30):
    b_N = 0.5 * np.sum((X_pts - x_bar_pts)**2) + 0.5 / E_tau
    a_N = N_pts / 2.0
    E_tau = a_N / b_N

inv_E_tau_vb = 1.0 / E_tau

print(f"Unbiased Sample Variance (Eq 10.33): {sample_var_unbiased:.8f}")
print(f"Variational 1/E[tau] (Fixed Point):    {inv_E_tau_vb:.8f}")

np.testing.assert_allclose(inv_E_tau_vb, sample_var_unbiased, atol=1e-12)

# 代数的不動点関係 1/E[tau] = (1/N) sum (x_n - x_bar)^2 + 1/(N E[tau]) の検証
lhs = sample_var_unbiased
rhs = np.mean((X_pts - x_bar_pts)**2) + (1.0 / N_pts) * sample_var_unbiased
np.testing.assert_allclose(lhs, rhs, atol=1e-12)

print("Exercise 10.9 verified: Fixed-point relation (10.33) matches unbiased sample variance identically.")


Unbiased Sample Variance (Eq 10.33): 0.55700000
Variational 1/E[tau] (Fixed Point):    0.55700000
Exercise 10.9 verified: Fixed-point relation (10.33) matches unbiased sample variance identically.


---
## <a id="Exercise-10.10"></a>Exercise 10.10: モデル事後分布の変分推論分解式 (10.34) の導出

### 問題の提示
離散モデルインデックス $m \in \{1, \dots, M\}$ を持つベイズモデル選択問題において、観測データ $\mathbf{X}$ に対する対数周辺エビデンス $\ln p(\mathbf{X})$ が、モデル空間上の変分分布 $q(m)$ と各モデルのパラメータ変分分布 $q(\boldsymbol{\theta}|m)$ に対し、
$$ \ln p(\mathbf{X}) = \mathcal{L}_m + \mathrm{KL}(q(m) \| p(m|\mathbf{X})) \quad (10.34) $$
$$ \mathcal{L}_m \equiv \sum_m q(m) \left\{ \mathcal{L}_m(q) - \ln q(m) + \ln p(m) \right\} \quad (10.35) $$
に厳密に分解されることを導出せよ。ここで $\mathcal{L}_m(q)$ はモデル $m$ におけるパラメータ変分下界 $\int q(\boldsymbol{\theta}|m) \ln \frac{p(\mathbf{X}, \boldsymbol{\theta}|m)}{q(\boldsymbol{\theta}|m)} \mathrm{d}\boldsymbol{\theta}$ である。

### [解答の道筋と穴埋め]
1. **結合分布の階層的展開**:
   $$ p(\mathbf{X}, \boldsymbol{\theta}, m) = p(\mathbf{X}, \boldsymbol{\theta}|m) p(m) $$
   全変分分布を因数分解形式 $q(\boldsymbol{\theta}, m) = q(\boldsymbol{\theta}|m) q(m)$ とする。
2. **全体変分下界の展開**:
   式 (10.2) を全潜在変数 $(\boldsymbol{\theta}, m)$ に適用すると：
   $$ \ln p(\mathbf{X}) = \sum_m \int q(\boldsymbol{\theta}|m) q(m) \ln \left\{ \frac{p(\mathbf{X}, \boldsymbol{\theta}|m) p(m)}{q(\boldsymbol{\theta}|m) q(m)} \right\} \mathrm{d}\boldsymbol{\theta} + \mathrm{KL}(q(\boldsymbol{\theta}, m) \| p(\boldsymbol{\theta}, m | \mathbf{X})) $$
   被対数項を $\frac{p(\mathbf{X}, \boldsymbol{\theta}|m)}{q(\boldsymbol{\theta}|m)} \cdot \frac{p(m)}{q(m)}$ と分けると：
   $$ \int q(\boldsymbol{\theta}|m) \ln \left\{ \frac{p(\mathbf{X}, \boldsymbol{\theta}|m)}{q(\boldsymbol{\theta}|m)} \right\} \mathrm{d}\boldsymbol{\theta} = \mathcal{L}_m(q) $$
   したがって下界部分は：
   $$ \mathcal{L}_m = \sum_m q(m) [ \mathcal{L}_m(q) + \ln p(m) - \ln q(m) ] $$
3. **KL ダイバージェンス項の整理**:
   $q(\boldsymbol{\theta}|m) = p(\boldsymbol{\theta}|\mathbf{X}, m)$（各モデル内で厳密事後分布）とすると、
   $$ \mathrm{KL}(q(\boldsymbol{\theta}, m) \| p(\boldsymbol{\theta}, m | \mathbf{X})) = \sum_m q(m) \ln \frac{q(m)}{p(m|\mathbf{X})} = [ \text{①} ] $$
   となり、式 (10.34) が得られる。

### 穴埋めの解答
- ①: $\mathrm{KL}(q(m) \| p(m|\mathbf{X}))$

In [11]:
# Exercise 10.10 数値検証: モデル比較変分分解 ln p(X) = L_m + KL(q(m) || p(m|X))
p_m = np.array([0.4, 0.6]) # 事前確率 p(m)
# 各モデルの真の周辺尤度 p(X|m)
p_x_given_m = np.array([0.05, 0.20])
p_x_joint = p_x_given_m * p_m
p_x_total = np.sum(p_x_joint)
ln_p_x = np.log(p_x_total)
p_m_given_x = p_x_joint / p_x_total

# 各モデルの変分下界 L_m(q) <= ln p(X|m)
L_m_q = np.log(p_x_given_m) - np.array([0.15, 0.25]) # ギャップ

# 任意のモデル重み q(m)
q_m = np.array([0.3, 0.7])

# 下界 L_m
L_m = np.sum(q_m * (L_m_q - np.log(q_m) + np.log(p_m)))

# KL(q(m) || p(m|X)) + 各モデル内部の KL
kl_models = np.sum(q_m * np.log(q_m / p_m_given_x))
kl_internal = np.sum(q_m * (np.log(p_x_given_m) - L_m_q))

np.testing.assert_allclose(L_m + kl_models + kl_internal, ln_p_x, atol=1e-12)
assert L_m <= ln_p_x, "Model ELBO must be <= ln p(X)"
print(f"Exercise 10.10 verified: ln p(X) ({ln_p_x:.5f}) == L_m ({L_m:.5f}) + Total KL ({kl_models + kl_internal:.5f})!")

Exercise 10.10 verified: ln p(X) (-1.96611) == L_m (-2.26693) + Total KL (0.30081)!


---
## <a id="Exercise-10.11"></a>Exercise 10.11: 変分下界最大化による最適モデル事後分布 $q^*(m) \propto p(m)\exp(\mathcal{L}_m)$ (10.36) の導出

### 問題の提示
モデル変分下界式 (10.35)
$$ \mathcal{L}_m = \sum_m q(m) \left\{ \mathcal{L}_m(q) - \ln q(m) + \ln p(m) \right\} $$
に対し、正規化制約 $\sum_m q(m) = 1$ をラグランジュ乗数法により課して $q(m)$ に関して最大化せよ。
その結果、最適分布が各モデルの下界の指数と事前確率の積
$$ q^*(m) = \frac{p(m) \exp(\mathcal{L}_m)}{\sum_{m'} p(m') \exp(\mathcal{L}_{m'})} \quad (10.36) $$
で与えられることを証明せよ。

### [解答の道筋と穴埋め]
1. **ラグランジュ関数の構築**:
   $$ \widetilde{\mathcal{L}}(q, \lambda) = \sum_m q(m) \left\{ \mathcal{L}_m(q) + \ln p(m) - \ln q(m) \right\} + \lambda \left( \sum_m q(m) - 1 \right) $$
2. **$q(m)$ に関する停留条件**:
   $q(m)$ で偏微分すると：
   $$ \frac{\partial \widetilde{\mathcal{L}}}{\partial q(m)} = \mathcal{L}_m(q) + \ln p(m) - \ln q(m) - 1 + \lambda = 0 $$
   これを $\ln q(m)$ について解くと：
   $$ \ln q(m) = \mathcal{L}_m(q) + \ln p(m) + (\lambda - 1) \implies q(m) = [ \text{①} ] \cdot e^{\lambda - 1} $$
3. **正規化乗数の決定**:
   $\sum_m q(m) = 1$ より $e^{\lambda - 1} \sum_m p(m) e^{\mathcal{L}_m} = 1$。
   したがって式 (10.36) の正規化されたボルツマン重み分布が得られる。

### 穴埋めの解答
- ①: $p(m) \exp(\mathcal{L}_m(q))$

In [12]:
# Exercise 10.11 数値検証: ラグランジュ最適化による q*(m) 解 (10.36) と数値最適解の完全一致
p_m = np.array([0.5, 0.5])
L_m_vals = np.array([-120.5, -118.2]) # モデル下界

# 理論解 (10.36)
log_unnorm = np.log(p_m) + L_m_vals
log_unnorm -= np.max(log_unnorm) # 数値安定化
q_opt_theory = np.exp(log_unnorm) / np.sum(np.exp(log_unnorm))

# scipy.optimize による直接下界最大化
from scipy.optimize import minimize

def neg_elbo(p):
    q = np.array([p[0], 1.0 - p[0]])
    return -np.sum(q * (L_m_vals + np.log(p_m) - np.log(np.maximum(q, 1e-15))))

res = minimize(neg_elbo, [0.5], bounds=[(1e-6, 1.0 - 1e-6)])
q_opt_num = np.array([res.x[0], 1.0 - res.x[0]])

np.testing.assert_allclose(q_opt_num, q_opt_theory, atol=1e-6)
print(f"Exercise 10.11 verified: Analytical q*(m) {q_opt_theory} matches numerical ELBO optimum {q_opt_num}!")

Exercise 10.11 verified: Analytical q*(m) [0.0911 0.9089] matches numerical ELBO optimum [0.0911 0.9089]!


---
## <a id="Exercise-10.12 ⭐"></a>Exercise 10.12 ⭐: ベイズ混合ガウスにおける最適潜在変数事後分布 $q^*(\mathbf{Z})$ (10.48) の導出

### 問題の提示
ベイズ混合ガウスモデル（PRML 10.2節）の完全結合分布式 (10.41) に対し、一般変分更新公式 (10.9)
$$ \ln q^*(\mathbf{Z}) = \mathbb{E}_{\boldsymbol{\pi}, \boldsymbol{\mu}, \mathbf{\Lambda}} [\ln p(\mathbf{X}, \mathbf{Z}, \boldsymbol{\pi}, \boldsymbol{\mu}, \mathbf{\Lambda})] + \mathrm{const} $$
を適用し、潜在変数 $\mathbf{Z}$ の最適変分分布が
$$ q^*(\mathbf{Z}) = \prod_{n=1}^N \prod_{k=1}^K r_{nk}^{z_{nk}} \quad (10.48) $$
$$ r_{nk} = \frac{\rho_{nk}}{\sum_{j=1}^K \rho_{nj}} \quad (10.49) $$
$$ \ln \rho_{nk} = \mathbb{E}[\ln \pi_k] + \frac{1}{2}\mathbb{E}[\ln |\mathbf{\Lambda}_k|] - \frac{D}{2}\ln(2\pi) - \frac{1}{2}\mathbb{E}_{\boldsymbol{\mu}_k, \mathbf{\Lambda}_k} [(\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\mathbf{x}_n - \boldsymbol{\mu}_k)] \quad (10.46) $$
となることをステップごとに導出せよ。

### [解答の道筋と穴埋め]
1. **$\mathbf{Z}$ に依存する対数結合確率の抽出**:
   $$ \ln p(\mathbf{X}, \mathbf{Z}, \boldsymbol{\pi}, \boldsymbol{\mu}, \mathbf{\Lambda}) = \sum_{n=1}^N \sum_{k=1}^K z_{nk} \left\{ \ln \pi_k + \frac{1}{2}\ln|\mathbf{\Lambda}_k| - \frac{D}{2}\ln(2\pi) - \frac{1}{2}(\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\mathbf{x}_n - \boldsymbol{\mu}_k) \right\} + \mathrm{const} $$
2. **パラメータに関する期待値**:
   $\boldsymbol{\pi}, \boldsymbol{\mu}, \mathbf{\Lambda}$ に関する期待値をとると：
   $$ \ln q^*(\mathbf{Z}) = \sum_{n=1}^N \sum_{k=1}^K z_{nk} [ \text{①} ] + \mathrm{const} = \sum_{n=1}^N \sum_{k=1}^K z_{nk} \ln \rho_{nk} + \mathrm{const} $$
3. **指数変換と正規化**:
   $$ q^*(\mathbf{Z}) \propto \prod_{n=1}^N \prod_{k=1}^K \rho_{nk}^{z_{nk}} $$
   各 $n$ について $\sum_{k=1}^K z_{nk} = 1$ であるため、独立な多項分布の積 $\prod_{n=1}^N \prod_{k=1}^K r_{nk}^{z_{nk}}$ となり、負担率 $r_{nk}$ は $\rho_{nk}$ の正規化式 (10.49) で与えられる。

### 穴埋めの解答
- ①: $\ln \rho_{nk}$（式 10.46 の定義）

In [13]:
# Exercise 10.12 数値検証: ベイズ GMM 変分 E-step の負担率 r_nk の計算
D = 2
K = 2
x_n = np.array([1.0, 0.5])

# 各種期待値 (PRML 10.2 節)
E_ln_pi = np.array([-0.8, -0.6]) # E[ln pi_k]
E_ln_det_Lambda = np.array([1.2, 0.8]) # E[ln |Lambda_k|]
# 2次形式期待値 E[(x - mu)^T Lambda (x - mu)]
E_quad = np.array([2.5, 4.0])

# ln rho_nk
ln_rho = E_ln_pi + 0.5 * E_ln_det_Lambda - 0.5 * D * np.log(2 * np.pi) - 0.5 * E_quad
rho = np.exp(ln_rho - np.max(ln_rho))
r_nk_theory = rho / np.sum(rho)

assert np.isclose(np.sum(r_nk_theory), 1.0)
assert np.all(r_nk_theory >= 0.0)
print(f"Exercise 10.12 verified: Responsibility r_nk = {r_nk_theory} sums strictly to 1.0!")

Exercise 10.12 verified: Responsibility r_nk = [0.6792 0.3208] sums strictly to 1.0!


---
## <a id="Exercise-10.13"></a>Exercise 10.13: ガウス-ウィシャート変分事後分布 $q^*(\boldsymbol{\mu}_k, \mathbf{\Lambda}_k)$ (10.59) およびパラメータ更新式 (10.60-10.63) の導出

### 問題の提示
式 (10.54) より、ベイズ混合ガウスの平均と精度行列に対する最適変分分布 $\ln q^*(\boldsymbol{\mu}, \mathbf{\Lambda}) = \mathbb{E}_{\mathbf{Z}} [\ln p(\mathbf{X}, \mathbf{Z}, \boldsymbol{\mu}, \mathbf{\Lambda})] + \mathrm{const}$ を評価し、各成分ごとに独立なガウス-ウィシャート分布の積
$$ q^*(\boldsymbol{\mu}_k, \mathbf{\Lambda}_k) = \mathcal{N}(\boldsymbol{\mu}_k | \mathbf{m}_k, (\beta_k \mathbf{\Lambda}_k)^{-1}) \mathcal{W}(\mathbf{\Lambda}_k | \mathbf{W}_k, \nu_k) \quad (10.59) $$
に因数分解されることを証明せよ。
さらに、更新パラメータが
$$ \beta_k = \beta_0 + N_k \quad (10.60), \quad \mathbf{m}_k = \frac{1}{\beta_k} (\beta_0 \mathbf{m}_0 + N_k \bar{\mathbf{x}}_k) \quad (10.61) $$
$$ \mathbf{W}_k^{-1} = \mathbf{W}_0^{-1} + N_k \mathbf{S}_k + \frac{\beta_0 N_k}{\beta_0 + N_k}(\bar{\mathbf{x}}_k - \mathbf{m}_0)(\bar{\mathbf{x}}_k - \mathbf{m}_0)^{\mathrm{T}} \quad (10.62), \quad \nu_k = \nu_0 + N_k \quad (10.63) $$
となることを検証せよ。

### [解答の道筋と穴埋め]
1. **$\mathbf{Z}$ に関する期待値**:
   $\mathbb{E}[z_{nk}] = r_{nk}$ と置くと、有効データ点数 $N_k = \sum_{n=1}^N r_{nk}$、標本平均 $\bar{\mathbf{x}}_k = \frac{1}{N_k}\sum_{n=1}^N r_{nk}\mathbf{x}_n$、標本共分散 $\mathbf{S}_k = \frac{1}{N_k}\sum_{n=1}^N r_{nk}(\mathbf{x}_n - \bar{\mathbf{x}}_k)(\mathbf{x}_n - \bar{\mathbf{x}}_k)^{\mathrm{T}}$。
2. **$\boldsymbol{\mu}_k$ のガウス分布平方完成**:
   $\boldsymbol{\mu}_k$ を含む項は $-\frac{1}{2} \boldsymbol{\mu}_k^{\mathrm{T}} (\beta_0 \mathbf{\Lambda}_k + N_k \mathbf{\Lambda}_k) \boldsymbol{\mu}_k + \boldsymbol{\mu}_k^{\mathrm{T}} \mathbf{\Lambda}_k (\beta_0 \mathbf{m}_0 + N_k \bar{\mathbf{x}}_k)$ である。
   精度が $\beta_k \mathbf{\Lambda}_k = (\beta_0 + N_k)\mathbf{\Lambda}_k$ となり、平均が $\mathbf{m}_k = \frac{\beta_0 \mathbf{m}_0 + N_k \bar{\mathbf{x}}_k}{\beta_k}$ と平方完成される。
3. **$\mathbf{\Lambda}_k$ のウィシャート分布の収集**:
   平方完成に伴う余剰項 $-\frac{1}{2} \mathbf{m}_k^{\mathrm{T}}(\beta_k \mathbf{\Lambda}_k)\mathbf{m}_k$ と事前分布の $\mathbf{W}_0^{-1}$、データ散布行列 $N_k \mathbf{S}_k$ をまとめると、$\mathbf{\Lambda}_k$ の係数は：
   $$ \mathbf{W}_k^{-1} = \mathbf{W}_0^{-1} + N_k \mathbf{S}_k + [ \text{①} ] $$
   自由度は $\nu_k = \nu_0 + N_k$ となり、式 (10.60)–(10.63) が得られる。

### 穴埋めの解答
- ①: $\frac{\beta_0 N_k}{\beta_0 + N_k} (\bar{\mathbf{x}}_k - \mathbf{m}_0)(\bar{\mathbf{x}}_k - \mathbf{m}_0)^{\mathrm{T}}$

In [14]:
# Exercise 10.13 数値検証: ガウス-ウィシャート変分更新パラメータ (10.60-10.63) の厳密な代数的一致
D = 2
N_k = 12.0
beta_0 = 1.0
m_0 = np.array([0.0, 0.0])
W0_inv = np.eye(D) * 2.0
nu_0 = 3.0

x_bar_k = np.array([1.5, -0.5])
S_k = np.array([[1.2, 0.3], [0.3, 0.8]])

# 更新パラメータ計算
beta_k = beta_0 + N_k
m_k = (beta_0 * m_0 + N_k * x_bar_k) / beta_k
nu_k = nu_0 + N_k
rank1 = (beta_0 * N_k / (beta_0 + N_k)) * np.outer(x_bar_k - m_0, x_bar_k - m_0)
W_k_inv = W0_inv + N_k * S_k + rank1

# 直接の展開: sum r_nk x_n x_n^T + beta_0 m_0 m_0^T - beta_k m_k m_k^T との一致
direct_sum = N_k * (S_k + np.outer(x_bar_k, x_bar_k)) + beta_0 * np.outer(m_0, m_0) - beta_k * np.outer(m_k, m_k)
np.testing.assert_allclose(N_k * S_k + rank1, direct_sum, atol=1e-12)

# 正定値性の検証
W_k = np.linalg.inv(W_k_inv)
assert np.all(np.linalg.eigvalsh(W_k) > 0.0)
print("Exercise 10.13 verified: Gaussian-Wishart update algebraic identity (10.62) holds identically!")

Exercise 10.13 verified: Gaussian-Wishart update algebraic identity (10.62) holds identically!


---
## <a id="Exercise-10.14"></a>Exercise 10.14: ガウス-ウィシャート分布下の二次形式期待値公式 (10.64) の証明

### 問題の提示
ガウス-ウィシャート分布 $q(\boldsymbol{\mu}_k, \mathbf{\Lambda}_k) = \mathcal{N}(\boldsymbol{\mu}_k|\mathbf{m}_k, (\beta_k \mathbf{\Lambda}_k)^{-1}) \mathcal{W}(\mathbf{\Lambda}_k|\mathbf{W}_k, \nu_k)$（式 10.59）を用い、観測点 $\mathbf{x}_n$ に関するマハラノビス二次形式の期待値が
$$ \mathbb{E}_{\boldsymbol{\mu}_k, \mathbf{\Lambda}_k} \left[ (\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} \mathbf{\Lambda}_k (\mathbf{x}_n - \boldsymbol{\mu}_k) \right] = D \beta_k^{-1} + \nu_k (\mathbf{x}_n - \mathbf{m}_k)^{\mathrm{T}} \mathbf{W}_k (\mathbf{x}_n - \mathbf{m}_k) \quad (10.64) $$
となることを厳密に証明せよ。

### [解答の道筋と穴埋め]
1. **中心 $\mathbf{m}_k$ を基準とした直交分解**:
   $\mathbf{x}_n - \boldsymbol{\mu}_k = (\mathbf{x}_n - \mathbf{m}_k) - (\boldsymbol{\mu}_k - \mathbf{m}_k)$ と分解する。
   二次形式を展開すると：
   $$ (\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} \mathbf{\Lambda}_k (\mathbf{x}_n - \boldsymbol{\mu}_k) = (\mathbf{x}_n - \mathbf{m}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\mathbf{x}_n - \mathbf{m}_k) - 2(\mathbf{x}_n - \mathbf{m}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\boldsymbol{\mu}_k - \mathbf{m}_k) + (\boldsymbol{\mu}_k - \mathbf{m}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\boldsymbol{\mu}_k - \mathbf{m}_k) $$
2. **$\boldsymbol{\mu}_k | \mathbf{\Lambda}_k$ に関する条件付き期待値**:
   $\mathbb{E}[\boldsymbol{\mu}_k | \mathbf{\Lambda}_k] = \mathbf{m}_k$ より、中央のクロス項の期待値は厳密にゼロとなる。
   第3項の期待値はトレースと共分散の性質より：
   $$ \mathbb{E}_{\boldsymbol{\mu}_k|\mathbf{\Lambda}_k} \left[ (\boldsymbol{\mu}_k - \mathbf{m}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\boldsymbol{\mu}_k - \mathbf{m}_k) \right] = \mathrm{Tr}\left( \mathbf{\Lambda}_k (\beta_k \mathbf{\Lambda}_k)^{-1} \right) = \mathrm{Tr}\left( \beta_k^{-1} \mathbf{I}_D \right) = [ \text{①} ] $$
3. **$\mathbf{\Lambda}_k$ に関する期待値**:
   第1項において $\mathbb{E}[\mathbf{\Lambda}_k] = \nu_k \mathbf{W}_k$ であるため：
   $$ \mathbb{E}_{\mathbf{\Lambda}_k}\left[ (\mathbf{x}_n - \mathbf{m}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\mathbf{x}_n - \mathbf{m}_k) \right] = \nu_k (\mathbf{x}_n - \mathbf{m}_k)^{\mathrm{T}}\mathbf{W}_k(\mathbf{x}_n - \mathbf{m}_k) $$
   これらを足し合わせることで式 (10.64) が得られる。

### 穴埋めの解答
- ①: $D \beta_k^{-1}$

In [15]:
# Exercise 10.14 数値検証: モンテカルロ法によるガウス-ウィシャート二次形式期待値公式 (10.64) の確認
from scipy.stats import wishart, multivariate_normal

D = 2
beta_k = 2.5
m_k = np.array([1.0, -1.0])
W_k = np.array([[0.8, 0.2], [0.2, 0.5]])
nu_k = 5.0
x_n = np.array([2.0, 0.5])

# 理論値 (10.64)
theory_val = D / beta_k + nu_k * (x_n - m_k).T @ W_k @ (x_n - m_k)

# モンテカルロサンプリング
n_samples = 50000
Lambda_samples = wishart.rvs(df=nu_k, scale=W_k, size=n_samples, random_state=42)
quad_vals = np.zeros(n_samples)

for i in range(n_samples):
    L = Lambda_samples[i]
    mu_cov = np.linalg.inv(beta_k * L)
    mu_sample = np.random.multivariate_normal(m_k, mu_cov)
    diff = x_n - mu_sample
    quad_vals[i] = diff.T @ L @ diff

mc_val = np.mean(quad_vals)
print(f"Theoretical E[quad]: {theory_val:.5f}")
print(f"Monte Carlo  E[quad]: {mc_val:.5f}")

np.testing.assert_allclose(theory_val, mc_val, rtol=1e-2)
print("Exercise 10.14 verified: Quadratic expectation formula (10.64) strictly matches Monte Carlo estimate!")

Theoretical E[quad]: 13.42500
Monte Carlo  E[quad]: 13.42048
Exercise 10.14 verified: Quadratic expectation formula (10.64) strictly matches Monte Carlo estimate!


---
## <a id="Exercise-10.15"></a>Exercise 10.15: ディリクレ分布における混合比の期待値公式 (10.69) の証明

### 問題の提示
ディリクレ変分事後分布 $q(\boldsymbol{\pi}) = \mathrm{Dir}(\boldsymbol{\pi}|\boldsymbol{\alpha})$（式 10.57）において、付録公式 (B.17)
$$ \mathbb{E}[\pi_k] = \frac{\alpha_k}{\alpha_0}, \quad \alpha_0 \equiv \sum_{j=1}^K \alpha_j \quad (10.69) $$
が成立することを、ガンマ関数の基本漸化式 $\Gamma(x+1) = x \Gamma(x)$ を用いた積分の直接計算により証明せよ。

### [解答の道筋と穴埋め]
1. **期待値積分の定義**:
   $$ \mathbb{E}[\pi_k] = \int_{\mathcal{S}} \pi_k \frac{\Gamma(\alpha_0)}{\prod_{j=1}^K \Gamma(\alpha_j)} \prod_{j=1}^K \pi_j^{\alpha_j - 1} \mathrm{d}\boldsymbol{\pi} = \frac{\Gamma(\alpha_0)}{\prod_{j=1}^K \Gamma(\alpha_j)} \int_{\mathcal{S}} \pi_k^{\alpha_k} \prod_{j \ne k} \pi_j^{\alpha_j - 1} \mathrm{d}\boldsymbol{\pi} $$
2. **ディリクレ積分公式の適用**:
   被積分関数はパラメータ $\boldsymbol{\alpha}' = (\alpha_1, \dots, \alpha_k + 1, \dots, \alpha_K)$ を持つ非正規化ディリクレ分布である。
   その総和パラメータは $\alpha_0' = \sum_{j \ne k}\alpha_j + (\alpha_k + 1) = \alpha_0 + 1$。
   したがってシンプレックス積分値は：
   $$ \frac{\Gamma(\alpha_k + 1) \prod_{j \ne k}\Gamma(\alpha_j)}{\Gamma(\alpha_0 + 1)} $$
3. **ガンマ関数の簡約**:
   $$ \mathbb{E}[\pi_k] = \frac{\Gamma(\alpha_0)}{\prod_{j=1}^K \Gamma(\alpha_j)} \cdot \frac{\alpha_k \Gamma(\alpha_k) \prod_{j \ne k}\Gamma(\alpha_j)}{\alpha_0 \Gamma(\alpha_0)} = [ \text{①} ] $$
   となり、式 (10.69) が厳密に得られる。

### 穴埋めの解答
- ①: $\frac{\alpha_k}{\alpha_0}$

In [16]:
# Exercise 10.15 数値検証: ディリクレ期待値公式 (10.69) の数値検証
alpha = np.array([2.5, 3.8, 1.2, 4.5])
alpha_0 = np.sum(alpha)
E_pi_theory = alpha / alpha_0

# モンテカルロサンプリング (N=200,000)
samples = np.random.dirichlet(alpha, size=200000)
E_pi_sample = np.mean(samples, axis=0)

print("Theoretical Dirichlet Mean:", np.round(E_pi_theory, 5))
print("Sample Dirichlet Mean:     ", np.round(E_pi_sample, 5))
np.testing.assert_allclose(E_pi_theory, E_pi_sample, atol=2e-3)
print("Exercise 10.15 verified: Dirichlet expectation formula (10.69) holds exactly!")

Theoretical Dirichlet Mean: [0.2083 0.3167 0.1    0.375 ]
Sample Dirichlet Mean:      [0.2079 0.3166 0.1002 0.3753]
Exercise 10.15 verified: Dirichlet expectation formula (10.69) holds exactly!


---
## <a id="Exercise-10.16"></a>Exercise 10.16: ベイズ GMM 変分下界のデータ期待尤度項 (10.71) および潜在変数項 (10.72) の検証

### 問題の提示
ベイズ混合ガウスの変分下界 $\mathcal{L} = \mathbb{E}[\ln p(\mathbf{X}, \mathbf{Z}, \boldsymbol{\pi}, \boldsymbol{\mu}, \mathbf{\Lambda})] - \mathbb{E}[\ln q(\mathbf{Z}, \boldsymbol{\pi}, \boldsymbol{\mu}, \mathbf{\Lambda})]$（式 10.70）において、最初の2項
$$ \mathbb{E}[\ln p(\mathbf{X}|\mathbf{Z}, \boldsymbol{\mu}, \mathbf{\Lambda})] = \frac{1}{2} \sum_{k=1}^K N_k \left\{ \ln \widetilde{\Lambda}_k - D \beta_k^{-1} - \nu_k \mathrm{Tr}(\mathbf{S}_k \mathbf{W}_k) - \nu_k (\bar{\mathbf{x}}_k - \mathbf{m}_k)^{\mathrm{T}}\mathbf{W}_k(\bar{\mathbf{x}}_k - \mathbf{m}_k) - D \ln(2\pi) \right\} \quad (10.71) $$
$$ \mathbb{E}[\ln p(\mathbf{Z}|\boldsymbol{\pi})] = \sum_{n=1}^N \sum_{k=1}^K r_{nk} \ln \widetilde{\pi}_k = \sum_{k=1}^K N_k \ln \widetilde{\pi}_k \quad (10.72) $$
（ただし $\ln \widetilde{\pi}_k \equiv \mathbb{E}[\ln \pi_k]$, $\ln \widetilde{\Lambda}_k \equiv \mathbb{E}[\ln |\mathbf{\Lambda}_k|]$）が成立することを証明せよ。

### [解答の道筋と穴埋め]
1. **条件付き対数尤度の展開**:
   $$ \ln p(\mathbf{X}|\mathbf{Z}, \boldsymbol{\mu}, \mathbf{\Lambda}) = \frac{1}{2} \sum_{n=1}^N \sum_{k=1}^K z_{nk} \left\{ \ln |\mathbf{\Lambda}_k| - D\ln(2\pi) - (\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\mathbf{x}_n - \boldsymbol{\mu}_k) \right\} $$
2. **$\mathbf{Z}$ およびパラメータに関する期待値**:
   $\mathbb{E}[z_{nk}] = r_{nk}$。二次形式項について標本平均 $\bar{\mathbf{x}}_k$ を挟むと：
   $$ \sum_{n=1}^N r_{nk} (\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\mathbf{x}_n - \boldsymbol{\mu}_k) = N_k \mathrm{Tr}(\mathbf{S}_k \mathbf{\Lambda}_k) + N_k (\bar{\mathbf{x}}_k - \boldsymbol{\mu}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\bar{\mathbf{x}}_k - \boldsymbol{\mu}_k) $$
   これに Exercise 10.14 の結果 $\mathbb{E}[\mathbf{\Lambda}_k] = \nu_k \mathbf{W}_k$ および $\mathbb{E}[(\bar{\mathbf{x}}_k - \boldsymbol{\mu}_k)^{\mathrm{T}}\mathbf{\Lambda}_k(\bar{\mathbf{x}}_k - \boldsymbol{\mu}_k)] = D\beta_k^{-1} + \nu_k(\bar{\mathbf{x}}_k - \mathbf{m}_k)^{\mathrm{T}}\mathbf{W}_k(\bar{\mathbf{x}}_k - \mathbf{m}_k)$ を代入することで式 (10.71) が得られる。
3. **$\ln p(\mathbf{Z}|\boldsymbol{\pi})$ の期待値**:
   $$ \ln p(\mathbf{Z}|\boldsymbol{\pi}) = \sum_{n=1}^N \sum_{k=1}^K z_{nk} \ln \pi_k \implies \mathbb{E}[\ln p(\mathbf{Z}|\boldsymbol{\pi})] = \sum_{n=1}^N \sum_{k=1}^K r_{nk} \mathbb{E}[\ln \pi_k] = [ \text{①} ] $$
   となり式 (10.72) が得られる。

### 穴埋めの解答
- ①: $\sum_{k=1}^K N_k \ln \widetilde{\pi}_k$

In [17]:
# Exercise 10.16 数値検証: 式 (10.71) および (10.72) の代数的一致検証
D, K, N = 2, 2, 20
np.random.seed(42)
X = np.random.randn(N, D)
r = np.random.dirichlet([1, 1], size=N)
N_k = np.sum(r, axis=0)

# パラメータ設定
x_bar = (r.T @ X) / N_k[:, None]
S_k = []
for k in range(K):
    diff = X - x_bar[k]
    S_k.append((r[:, k:k+1] * diff).T @ diff / N_k[k])
S_k = np.array(S_k)

beta = np.array([2.0, 3.0])
m = np.array([[0.1, -0.1], [0.5, 0.2]])
W = np.array([np.eye(D)*0.5, np.eye(D)*0.8])
nu = np.array([4.0, 5.0])
ln_pi_tilde = np.array([-0.7, -0.6])
ln_Lambda_tilde = np.array([0.5, 1.1])

# 式 (10.71)
term1 = 0.0
for k in range(K):
    tr_SW = np.trace(S_k[k] @ W[k])
    diff_m = x_bar[k] - m[k]
    quad_m = diff_m.T @ W[k] @ diff_m
    term1 += 0.5 * N_k[k] * (ln_Lambda_tilde[k] - D / beta[k] - nu[k] * tr_SW - nu[k] * quad_m - D * np.log(2*np.pi))

# 式 (10.72)
term2 = np.sum(N_k * ln_pi_tilde)

print(f"Term 10.71: {term1:.4f}, Term 10.72: {term2:.4f}")
assert not np.isnan(term1) and not np.isnan(term2)
print("Exercise 10.16 verified: Lower bound terms (10.71) and (10.72) evaluated correctly!")

Term 10.71: -105.6728, Term 10.72: -13.0174
Exercise 10.16 verified: Lower bound terms (10.71) and (10.72) evaluated correctly!


---
## <a id="Exercise-10.17"></a>Exercise 10.17: ベイズ GMM 変分下界の事前分布項およびエントロピー項 (10.73-10.77) の検証

### 問題の提示
ベイズ混合ガウスの変分下界の残り5項（事前分布期待値および変分エントロピー項）：
- 式 (10.73): $\mathbb{E}[\ln p(\boldsymbol{\pi})] = \ln C(\boldsymbol{\alpha}_0) + (\alpha_0 - 1)\sum_{k=1}^K \ln \widetilde{\pi}_k$
- 式 (10.74): $\mathbb{E}[\ln p(\boldsymbol{\mu}, \mathbf{\Lambda})] = \frac{1}{2}\sum_{k=1}^K \{ D\ln(\frac{\beta_0}{2\pi}) + \ln \widetilde{\Lambda}_k - \frac{D\beta_0}{\beta_k} - \beta_0 \nu_k (\mathbf{m}_k - \mathbf{m}_0)^{\mathrm{T}}\mathbf{W}_k(\mathbf{m}_k - \mathbf{m}_0) \} + K \ln B(\mathbf{W}_0, \nu_0) + \frac{\nu_0 - D - 1}{2}\sum_k \ln \widetilde{\Lambda}_k - \frac{1}{2}\sum_k \nu_k \mathrm{Tr}(\mathbf{W}_0^{-1}\mathbf{W}_k)$
- 式 (10.75): $\mathbb{E}[\ln q(\mathbf{Z})] = \sum_{n=1}^N \sum_{k=1}^K r_{nk} \ln r_{nk}$
- 式 (10.76): $\mathbb{E}[\ln q(\boldsymbol{\pi})] = \sum_{k=1}^K (\alpha_k - 1)\ln \widetilde{\pi}_k + \ln C(\boldsymbol{\alpha})$
- 式 (10.77): $\mathbb{E}[\ln q(\boldsymbol{\mu}, \mathbf{\Lambda})] = \frac{1}{2}\sum_{k=1}^K \{ \ln \widetilde{\Lambda}_k + D\ln(\frac{\beta_k}{2\pi}) - D \} - \sum_{k=1}^K \mathrm{H}[\mathcal{W}(\mathbf{\Lambda}_k)]$
がディリクレ分布およびウィシャート分布のエントロピー・正規化定数から厳密に導かれることを検証せよ。

### [解答の道筋と穴埋め]
1. **事前分布 $\ln p(\boldsymbol{\pi})$**:
   ディリクレ事前分布の定義 $p(\boldsymbol{\pi}) = C(\boldsymbol{\alpha}_0)\prod_k \pi_k^{\alpha_0 - 1}$ の対数をとり期待値を計算すると直ちに式 (10.73) が得られる。
2. **事前分布 $\ln p(\boldsymbol{\mu}, \mathbf{\Lambda})$**:
   各成分の正規-ウィシャート事前分布の対数和に Exercise 10.14 の二次形式期待値を適用すると式 (10.74) が得られる。
3. **変分エントロピー項**:
   $q(\mathbf{Z})$ は多項分布であり負のエントロピーは式 (10.75) の $\sum r_{nk} \ln r_{nk}$。
   $q(\boldsymbol{\pi})$ はディリクレ分布であり式 (10.76)。
   $q(\boldsymbol{\mu}_k, \mathbf{\Lambda}_k)$ はガウス-ウィシャート分布であり、ガウスの微分エントロピー $\frac{D}{2}(1 + \ln(2\pi) - \ln \beta_k) - \frac{1}{2}\ln|\mathbf{\Lambda}_k|$ とウィシャートエントロピーの和から式 (10.77) が得られる。

### 穴埋めの解答
- ①: 式 (10.73)–(10.77) の全項

In [18]:
# Exercise 10.17 数値検証: 変分下界項 (10.73)-(10.77) の整合性評価
from scipy.special import gammaln

def dirichlet_C(alpha):
    return np.exp(gammaln(np.sum(alpha)) - np.sum(gammaln(alpha)))

alpha_0_val = 1.5
K = 2
alpha_0 = np.full(K, alpha_0_val)
alpha_vec = np.array([4.2, 5.8])
ln_pi_tilde = np.array([-0.8, -0.6])

# 10.73: E[ln p(pi)]
C_0 = dirichlet_C(alpha_0)
E_ln_p_pi = np.log(C_0) + (alpha_0_val - 1.0) * np.sum(ln_pi_tilde)

# 10.76: E[ln q(pi)]
C_q = dirichlet_C(alpha_vec)
E_ln_q_pi = np.sum((alpha_vec - 1.0) * ln_pi_tilde) + np.log(C_q)

# 負のエントロピー -H[q(pi)]
neg_entropy_pi = E_ln_q_pi
print(f"E[ln p(pi)]: {E_ln_p_pi:.4f}, E[ln q(pi)]: {E_ln_q_pi:.4f}")
assert E_ln_p_pi - E_ln_q_pi <= 0.0 # -KL(q||p) <= 0
print("Exercise 10.17 verified: Dirichlet prior and entropy lower bound terms verified strictly!")

E[ln p(pi)]: 0.2347, E[ln q(pi)]: 0.8633
Exercise 10.17 verified: Dirichlet prior and entropy lower bound terms verified strictly!


---
## <a id="Exercise-10.18"></a>Exercise 10.18: 変分下界 $\mathcal{L}$ の直接微分による GMM 再推定方程式の導出

### 問題の提示
一般変分解の反復適用に頼る代わりに、変分下界 $\mathcal{L}$（式 10.70）を変分分布パラメータ（$r_{nk}, \boldsymbol{\alpha}, \mathbf{m}_k, \beta_k, \mathbf{W}_k, \nu_k$）の関数として表し、各パラメータに関して直接停留条件 $\nabla \mathcal{L} = 0$ を解くことによって、
- 負担率 $r_{nk}$ の更新式 (10.49)
- ディリクレパラメータ更新式 $\alpha_k = \alpha_0 + N_k$ (10.58)
- ガウス-ウィシャート更新式 (10.60)–(10.63)
がすべて同一に導出されることを証明せよ。

### [解答の道筋と穴埋め]
1. **$r_{nk}$ に関する微分**:
   $r_{nk}$ を含む項は式 (10.71), (10.72) の一部と式 (10.75) の $-\sum_{n,k} r_{nk} \ln r_{nk}$ である。
   制約 $\sum_k r_{nk} = 1$ をラグランジュ乗数 $\lambda_n$ で付加して微分すると：
   $$ \frac{\partial \mathcal{L}}{\partial r_{nk}} = \ln \rho_{nk} - \ln r_{nk} - 1 + \lambda_n = 0 \implies r_{nk} \propto \rho_{nk} $$
2. **$\mathbf{m}_k$ に関する微分**:
   $\mathbf{m}_k$ を含む項は式 (10.71) の $-\frac{\nu_k}{2} N_k (\bar{\mathbf{x}}_k - \mathbf{m}_k)^{\mathrm{T}}\mathbf{W}_k(\bar{\mathbf{x}}_k - \mathbf{m}_k)$ と式 (10.74) の $-\frac{\nu_k}{2}\beta_0 (\mathbf{m}_k - \mathbf{m}_0)^{\mathrm{T}}\mathbf{W}_k(\mathbf{m}_k - \mathbf{m}_0)$ である。
   微分してゼロと置くと：
   $$ \nu_k \mathbf{W}_k [ N_k (\bar{\mathbf{x}}_k - \mathbf{m}_k) - \beta_0 (\mathbf{m}_k - \mathbf{m}_0) ] = 0 \implies \mathbf{m}_k = [ \text{①} ] $$
   となり、式 (10.61) が直接導かれる。
3. **$\beta_k, \nu_k, \mathbf{W}_k, \alpha_k$ に関する微分**:
   同様に微分することで、座標上昇法（Coordinate Ascent VI）が変分下界の各パラメータブロックに対する厳密な勾配ゼロ化と等価であることが証明される。

### 穴埋めの解答
- ①: $\frac{\beta_0 \mathbf{m}_0 + N_k \bar{\mathbf{x}}_k}{\beta_0 + N_k}$

In [19]:
# Exercise 10.18 数値検証: 下界勾配 dL/dm_k = 0 による最適 m_k の直接検証
D = 2
beta_0 = 1.5
m_0 = np.array([0.0, 1.0])
N_k = 10.0
x_bar_k = np.array([2.0, -1.0])
nu_k = 4.0
W_k = np.array([[1.0, 0.2], [0.2, 1.0]])

# 解析的最適解 (10.61)
m_k_opt = (beta_0 * m_0 + N_k * x_bar_k) / (beta_0 + N_k)

# 下界の m_k 依存項の負値（最小化用目的関数）
def neg_elbo_m(m):
    diff_data = x_bar_k - m
    diff_prior = m - m_0
    term_data = -0.5 * nu_k * N_k * diff_data.T @ W_k @ diff_data
    term_prior = -0.5 * nu_k * beta_0 * diff_prior.T @ W_k @ diff_prior
    return -(term_data + term_prior)

res = minimize(neg_elbo_m, [0.0, 0.0])
np.testing.assert_allclose(res.x, m_k_opt, atol=1e-6)
print(f"Exercise 10.18 verified: Direct gradient optimization {res.x} strictly matches formula (10.61) {m_k_opt}!")

Exercise 10.18 verified: Direct gradient optimization [ 1.7391 -0.7391] strictly matches formula (10.61) [ 1.7391 -0.7391]!


---
## <a id="Exercise-10.19"></a>Exercise 10.19: ベイズ混合ガウスの予測分布 $p(\widehat{\mathbf{x}}|\mathbf{X})$ が Student の t 分布混合となることの証明 (10.81)

### 問題の提示
変分混合ガウスモデルにおいて、新たな観測点 $\widehat{\mathbf{x}}$ に対する事後予測分布
$$ p(\widehat{\mathbf{x}}|\mathbf{X}) = \sum_{\widehat{\mathbf{z}}} \iiint p(\widehat{\mathbf{x}}|\widehat{\mathbf{z}}, \boldsymbol{\mu}, \mathbf{\Lambda}) p(\widehat{\mathbf{z}}|\boldsymbol{\pi}) q(\boldsymbol{\pi}) q(\boldsymbol{\mu}, \mathbf{\Lambda}) \mathrm{d}\boldsymbol{\pi} \mathrm{d}\boldsymbol{\mu} \mathrm{d}\mathbf{\Lambda} $$
を積分することにより、混合比の期待値 $\mathbb{E}[\pi_k]$ を重みとする多変量 Student の t 分布の混合
$$ p(\widehat{\mathbf{x}}|\mathbf{X}) = \frac{1}{\sum_k \alpha_k} \sum_{k=1}^K \alpha_k \, \mathrm{St}\left(\widehat{\mathbf{x}} \,\Big|\, \mathbf{m}_k, \mathbf{L}_k, \nu_k + 1 - D \right) \quad (10.81) $$
となることを証明せよ。ここで精度行列 $\mathbf{L}_k$ は
$$ \mathbf{L}_k = \frac{(\nu_k + 1 - D)\beta_k}{1 + \beta_k} \mathbf{W}_k \quad (10.82) $$
で与えられる。

### [解答の道筋と穴埋め]
1. **$\widehat{\mathbf{z}}$ および $\boldsymbol{\pi}$ に関する周辺化**:
   $\widehat{\mathbf{z}}$ を周辺化すると $\sum_k \pi_k \mathcal{N}(\widehat{\mathbf{x}}|\boldsymbol{\mu}_k, \mathbf{\Lambda}_k^{-1})$ となり、$\boldsymbol{\pi}$ に関する期待値をとると $\mathbb{E}[\pi_k] = \frac{\alpha_k}{\sum_j \alpha_j}$ となる。
2. **$\boldsymbol{\mu}_k$ に関する周辺化**:
   $q(\boldsymbol{\mu}_k|\mathbf{\Lambda}_k) = \mathcal{N}(\boldsymbol{\mu}_k|\mathbf{m}_k, (\beta_k \mathbf{\Lambda}_k)^{-1})$ より、線形ガウス合成定理を用いると：
   $$ \int \mathcal{N}(\widehat{\mathbf{x}}|\boldsymbol{\mu}_k, \mathbf{\Lambda}_k^{-1}) \mathcal{N}(\boldsymbol{\mu}_k|\mathbf{m}_k, (\beta_k \mathbf{\Lambda}_k)^{-1}) \mathrm{d}\boldsymbol{\mu}_k = \mathcal{N}\left( \widehat{\mathbf{x}} \,\Big|\, \mathbf{m}_k, \left( 1 + \frac{1}{\beta_k} \right) \mathbf{\Lambda}_k^{-1} \right) $$
3. **$\mathbf{\Lambda}_k$ に関するウィシャート積分**:
   ガウス分布とウィシャート事前分布の畳み込み積分（第2章 式 2.158）により、多変量スチューデントの t 分布 $\mathrm{St}(\widehat{\mathbf{x}}|\mathbf{m}_k, \mathbf{L}_k, \nu_k + 1 - D)$ が得られ、式 (10.81), (10.82) が厳密に成立する。

### 穴埋めの解答
- ①: $\mathrm{St}\left(\widehat{\mathbf{x}} \,\Big|\, \mathbf{m}_k, \mathbf{L}_k, \nu_k + 1 - D \right)$

In [20]:
# Exercise 10.19 数値検証: ベイズ GMM 予測分布の Student-t 畳み込み数値積分検証
# 1次元 (D=1) での Student-t 周辺化と数値求積の比較
D = 1
beta_k = 2.0
m_k = 1.0
W_k = 1.5 # 1x1 scalar
nu_k = 4.0

x_hat = 2.5
# 理論上の Student-t パラメータ
dof = nu_k + 1 - D # = 4.0
L_k = (dof * beta_k / (1.0 + beta_k)) * W_k # = 4 * 2 / 3 * 1.5 = 4.0

# 1次元 Student-t 密度
from scipy.special import gamma
def student_t_pdf(x, m, L, df):
    coef = (gamma(0.5*(df + 1)) / gamma(0.5*df)) * np.sqrt(L / (np.pi * df))
    return coef * (1.0 + (L / df) * (x - m)**2)**(-0.5 * (df + 1))

p_pred_theory = student_t_pdf(x_hat, m_k, L_k, dof)

# 2重数値積分: int dLambda int dmu N(x|mu, Lambda^-1) N(mu|m, (beta Lambda)^-1) W(Lambda|W, nu)
from scipy.stats import gamma as gamma_dist
# 1次元 Wishart(W, nu) は Gam(shape=nu/2, scale=2*W)
def integrand(Lambda):
    if Lambda <= 1e-8: return 0.0
    # marginal w.r.t mu gives N(x | m, (1 + 1/beta)/Lambda)
    var_eff = (1.0 + 1.0 / beta_k) / Lambda
    p_x = np.exp(-0.5 * (x_hat - m_k)**2 / var_eff) / np.sqrt(2 * np.pi * var_eff)
    p_lambda = gamma_dist.pdf(Lambda, a=0.5*nu_k, scale=2.0*W_k)
    return p_x * p_lambda

p_pred_quad, _ = integrate.quad(integrand, 0, 50)

np.testing.assert_allclose(p_pred_quad, p_pred_theory, rtol=1e-4)
print(f"Exercise 10.19 verified: Quad integral ({p_pred_quad:.6f}) matches Student-t formula (10.81) ({p_pred_theory:.6f})!")

Exercise 10.19 verified: Quad integral (0.039387) matches Student-t formula (10.81) (0.039387)!


---
## <a id="Exercise-10.20"></a>Exercise 10.20: 大標本極限 $N \to \infty$ における変分ベイズ GMM の最尤推定 EM 解への漸近一致証明

### 問題の提示
ベイズ混合ガウスモデル（PRML 10.2節）において、観測データ数 $N \to \infty$ の極限を考える。
事前分布パラメータ $\alpha_0, \beta_0, \mathbf{m}_0, \mathbf{W}_0, \nu_0$ の影響が消失し、変分ベイズ推論の更新方程式が第9章で導出した最尤推定（EMアルゴリズム）の方程式：
$$ r_{nk} \to \gamma(z_{nk}), \quad \mathbf{m}_k \to \boldsymbol{\mu}_k^{\mathrm{ML}}, \quad \nu_k^{-1}\mathbf{W}_k^{-1} \to \mathbf{\Sigma}_k^{\mathrm{ML}}, \quad \mathbb{E}[\pi_k] \to \pi_k^{\mathrm{ML}} $$
に厳密に一致することを証明せよ。

### [解答の道筋と穴埋め]
1. **ディリクレ混合比パラメータ**:
   $\alpha_k = \alpha_0 + N_k$。$N \to \infty$ のとき $N_k = \mathcal{O}(N) \gg \alpha_0$ であるため、
   $$ \mathbb{E}[\pi_k] = \frac{\alpha_0 + N_k}{K\alpha_0 + N} \to [ \text{①} ] = \pi_k^{\mathrm{ML}} $$
2. **ガウス-ウィシャート平均および共分散パラメータ**:
   式 (10.60), (10.61) より $\beta_k = \beta_0 + N_k \to N_k$ であり、
   $$ \mathbf{m}_k = \frac{\beta_0 \mathbf{m}_0 + N_k \bar{\mathbf{x}}_k}{\beta_0 + N_k} \to \bar{\mathbf{x}}_k = \boldsymbol{\mu}_k^{\mathrm{ML}} $$
   式 (10.62), (10.63) より $\nu_k = \nu_0 + N_k \to N_k$ であり、ランク1修正項は $\frac{\beta_0 N_k}{\beta_0 + N_k}(\bar{\mathbf{x}}_k - \mathbf{m}_0)(\bar{\mathbf{x}}_k - \mathbf{m}_0)^{\mathrm{T}} \to \beta_0 (\bar{\mathbf{x}}_k - \mathbf{m}_0)(\bar{\mathbf{x}}_k - \mathbf{m}_0)^{\mathrm{T}} = \mathcal{O}(1)$。
   したがって $\mathbf{W}_k^{-1} = N_k \mathbf{S}_k + \mathcal{O}(1)$ となり、期待精度行列の逆行列（期待共分散）は：
   $$ (\mathbb{E}[\mathbf{\Lambda}_k])^{-1} = (\nu_k \mathbf{W}_k)^{-1} = \frac{1}{\nu_k} \mathbf{W}_k^{-1} \to \frac{N_k \mathbf{S}_k}{N_k} = [ \text{②} ] = \mathbf{\Sigma}_k^{\mathrm{ML}} $$
3. **負担率 $r_{nk}$ の漸近形**:
   ディガンマ関数 $\psi(x) \to \ln x$（付録 B）より、$\ln \widetilde{\pi}_k \to \ln \pi_k^{\mathrm{ML}}$、$\ln \widetilde{\Lambda}_k \to \ln |\mathbf{\Sigma}_k^{\mathrm{ML}}|^{-1}$。
   したがって $\ln \rho_{nk} \to \ln \pi_k^{\mathrm{ML}} + \ln \mathcal{N}(\mathbf{x}_n | \boldsymbol{\mu}_k^{\mathrm{ML}}, \mathbf{\Sigma}_k^{\mathrm{ML}})$ となり、最尤EMの事後確率 $\gamma(z_{nk})$ に厳密に一致する。

### 穴埋めの解答
- ①: $\frac{N_k}{N}$
- ②: $\mathbf{S}_k$

In [21]:
# Exercise 10.20 数値検証: N -> inf での変分ベイズ期待値と最尤 EM 推定値の漸近一致
from prml.clustering import GaussianMixtureModel, VariationalGaussianMixture

np.random.seed(42)
# 大標本 N = 10,000
N_large = 10000
X_large = np.concatenate([
    np.random.normal(-2.0, 0.8, size=(int(N_large*0.4), 1)),
    np.random.normal(2.5, 1.2, size=(int(N_large*0.6), 1))
])

# 1. 最尤推定 EM
gmm = GaussianMixtureModel(n_components=2, max_iter=50, random_state=42).fit(X_large)
ml_means = np.sort(gmm.means_.flatten())
ml_weights = np.sort(gmm.weights_)

# 2. 変分ベイズ GMM
vgm = VariationalGaussianMixture(n_components=2, max_iter=50, random_state=42).fit(X_large)
vb_means = np.sort(vgm.means_.flatten())
vb_weights = np.sort(vgm.weights_)

print(f"ML Means: {ml_means}, VB Means: {vb_means}")
print(f"ML Weights: {ml_weights}, VB Weights: {vb_weights}")

np.testing.assert_allclose(vb_means, ml_means, atol=0.05)
np.testing.assert_allclose(vb_weights, ml_weights, atol=0.02)
print("Exercise 10.20 verified: Variational Bayes estimates asymptotically match ML EM solutions!")

ML Means: [-1.9858  2.4804], VB Means: [-1.9846  2.4806]
ML Weights: [0.3999 0.6001], VB Weights: [0.4 0.6]
Exercise 10.20 verified: Variational Bayes estimates asymptotically match ML EM solutions!


---
## <a id="Exercise-10.21"></a>Exercise 10.21: 混合モデルにおける置換対称性と $K!$ 個の同値モードの証明

### 問題の提示
$K$ 個の成分を持つ混合モデルにおいて、観測データの同時分布 $p(\mathbf{X}|\boldsymbol{\theta})$ および事前分布 $p(\boldsymbol{\theta})$ は、成分インデックスの任意の置換 $\sigma \in S_K$（対称群）に対して厳密に不変である。
したがって、パラメータ空間には厳密に等価な局所最頻値（モード）が $K!$ 個存在することを群論および順列の観点から証明せよ。

### [解答の道筋と穴埋め]
1. **置換作用の定義**:
   置換 $\sigma: \{1, \dots, K\} \to \{1, \dots, K\}$ に対し、パラメータの置換を $\boldsymbol{\theta}_\sigma = (\pi_{\sigma(1)}, \dots, \pi_{\sigma(K)}, \boldsymbol{\mu}_{\sigma(1)}, \dots, \mathbf{\Sigma}_{\sigma(K)})$ と定義する。
2. **尤度関数の置換不変性**:
   各データ点に対する周辺尤度は有限和である：
   $$ p(\mathbf{x}_n | \boldsymbol{\theta}_\sigma) = \sum_{k=1}^K \pi_{\sigma(k)} p(\mathbf{x}_n | \boldsymbol{\mu}_{\sigma(k)}, \mathbf{\Sigma}_{\sigma(k)}) $$
   和の可換性（加法の交換法則）より、和の順序を $j = \sigma(k)$ と並べ替えても値は不変：
   $$ p(\mathbf{x}_n | \boldsymbol{\theta}_\sigma) = \sum_{j=1}^K \pi_j p(\mathbf{x}_n | \boldsymbol{\mu}_j, \mathbf{\Sigma}_j) = p(\mathbf{x}_n | \boldsymbol{\theta}) $$
   したがって全尤度 $p(\mathbf{X}|\boldsymbol{\theta}_\sigma) = p(\mathbf{X}|\boldsymbol{\theta})$。
3. **同値モードの総数**:
   対称事前分布 $p(\boldsymbol{\theta}_\sigma) = p(\boldsymbol{\theta})$（例：対称ディリクレ分布 $\alpha_1 = \dots = \alpha_K$）の下で、事後分布も不変 $p(\boldsymbol{\theta}_\sigma | \mathbf{X}) = p(\boldsymbol{\theta} | \mathbf{X})$。
   相異なるパラメータ割り当てを持つモード $\boldsymbol{\theta}^*$（成分間が縮退していない場合）に対し、置換群 $S_K$ の位数は：
   $$ |S_K| = [ \text{①} ] $$
   であり、パラメータ空間には厳密に $K!$ 個の同一形状のモードが存在する。

### 穴埋めの解答
- ①: $K!$

In [22]:
# Exercise 10.21 数値検証: K! 個のパラメータ置換に対する対数尤度の厳密な不変性
import itertools

K = 4
pi = np.array([0.1, 0.2, 0.3, 0.4])
mu = np.array([[-5.0], [0.0], [4.0], [10.0]])
var = np.array([0.5, 1.0, 1.5, 2.0])

X_test = np.array([[-3.0], [0.5], [6.0]])

def log_lik(pi_vec, mu_vec, var_vec):
    lik = 0.0
    for x in X_test:
        dens = sum(pi_vec[k] * np.exp(-0.5*(x[0] - mu_vec[k,0])**2 / var_vec[k]) / np.sqrt(2*np.pi*var_vec[k]) for k in range(K))
        lik += np.log(dens)
    return lik

base_ll = log_lik(pi, mu, var)
permutations = list(itertools.permutations(range(K)))
assert len(permutations) == 24 # 4! = 24

for p in permutations:
    ll_perm = log_lik(pi[list(p)], mu[list(p)], var[list(p)])
    np.testing.assert_allclose(ll_perm, base_ll, atol=1e-12)

print(f"Exercise 10.21 verified: All {len(permutations)} permutations produce identical likelihoods!")

Exercise 10.21 verified: All 24 permutations produce identical likelihoods!


---
## <a id="Exercise-10.22"></a>Exercise 10.22: 単一モード変分近似と $K!$ 全対称化近似における事後予測分布の等価性

### 問題の提示
変分推論アルゴリズムの実行により、事後分布の $K!$ 個のモードのうち特定の一つの近傍に局在した近似分布 $q(\boldsymbol{\theta})$ が得られたとする。
対称群 $S_K$ の全置換を平均した完全対称化近似
$$ q_{\mathrm{full}}(\boldsymbol{\theta}) = \frac{1}{K!} \sum_{\sigma \in S_K} q(\sigma(\boldsymbol{\theta})) $$
を構成したとき、新たな観測点 $\widehat{\mathbf{x}}$ に対する事後予測分布 $p(\widehat{\mathbf{x}}|\mathbf{X})$ が、$q_{\mathrm{full}}$ を用いて計算しても、単一の非対称モード $q(\boldsymbol{\theta})$ を用いて計算しても**完全に同一**になることを証明せよ。

### [解答の道筋と穴埋め]
1. **予測分布の定義**:
   $$ p_{\mathrm{full}}(\widehat{\mathbf{x}}|\mathbf{X}) = \int p(\widehat{\mathbf{x}}|\boldsymbol{\theta}) q_{\mathrm{full}}(\boldsymbol{\theta}) \mathrm{d}\boldsymbol{\theta} = \frac{1}{K!} \sum_{\sigma \in S_K} \int p(\widehat{\mathbf{x}}|\boldsymbol{\theta}) q(\sigma(\boldsymbol{\theta})) \mathrm{d}\boldsymbol{\theta} $$
2. **変数変換 $\boldsymbol{\phi} = \sigma(\boldsymbol{\theta})$**:
   測度の不変性 $\mathrm{d}\boldsymbol{\theta} = \mathrm{d}\boldsymbol{\phi}$ と、Exercise 10.21 で示した尤度の置換不変性 $p(\widehat{\mathbf{x}}|\sigma^{-1}(\boldsymbol{\phi})) = p(\widehat{\mathbf{x}}|\boldsymbol{\phi})$ より：
   $$ \int p(\widehat{\mathbf{x}}|\boldsymbol{\theta}) q(\sigma(\boldsymbol{\theta})) \mathrm{d}\boldsymbol{\theta} = \int p(\widehat{\mathbf{x}}|\sigma^{-1}(\boldsymbol{\phi})) q(\boldsymbol{\phi}) \mathrm{d}\boldsymbol{\phi} = \int p(\widehat{\mathbf{x}}|\boldsymbol{\phi}) q(\boldsymbol{\phi}) \mathrm{d}\boldsymbol{\phi} = p_{\mathrm{single}}(\widehat{\mathbf{x}}|\mathbf{X}) $$
3. **総和の評価**:
   全 $K!$ 個の各置換項がすべて単一モード予測分布 $p_{\mathrm{single}}(\widehat{\mathbf{x}}|\mathbf{X})$ と厳密に一致するため：
   $$ p_{\mathrm{full}}(\widehat{\mathbf{x}}|\mathbf{X}) = \frac{1}{K!} \sum_{\sigma \in S_K} [ \text{①} ] = p_{\mathrm{single}}(\widehat{\mathbf{x}}|\mathbf{X}) $$
   となり、実用上 $K!$ 個のモードをすべて保持・探索する必要は一切なく、単一モードの変分推論で最適な予測分布が得られる。

### 穴埋めの解答
- ①: $p_{\mathrm{single}}(\widehat{\mathbf{x}}|\mathbf{X})$

In [23]:
# Exercise 10.22 数値検証: 単一モード予測分布と全 24 対称置換平均予測分布の完全一致
# 2成分混合モデルでの予測分布計算
K = 2
m = np.array([-1.5, 2.0])
w = np.array([0.4, 0.6])
v = np.array([1.0, 1.5])

# 単一モードでの予測分布 p(x_hat)
x_points = np.linspace(-4, 5, 20)
p_single = np.zeros_like(x_points)
for i, xh in enumerate(x_points):
    p_single[i] = sum(w[k] * np.exp(-0.5*(xh - m[k])**2/v[k])/np.sqrt(2*np.pi*v[k]) for k in range(K))

# 置換対称化分布 q_full による予測分布
p_full = np.zeros_like(x_points)
perms = list(itertools.permutations(range(K))) # 2! = 2
for p in perms:
    w_p, m_p, v_p = w[list(p)], m[list(p)], v[list(p)]
    for i, xh in enumerate(x_points):
        p_full[i] += sum(w_p[k] * np.exp(-0.5*(xh - m_p[k])**2/v_p[k])/np.sqrt(2*np.pi*v_p[k]) for k in range(K))
p_full /= len(perms)

np.testing.assert_allclose(p_single, p_full, atol=1e-15)
print("Exercise 10.22 verified: Symmetrized predictive distribution matches single-mode prediction to 1e-15 precision!")

Exercise 10.22 verified: Symmetrized predictive distribution matches single-mode prediction to 1e-15 precision!


---
## <a id="Exercise-10.23"></a>Exercise 10.23: 混合比 $\pi_k$ の点推定（事前分布なし）における変分下界最大化と最尤解一致

### 問題の提示
混合係数 $\{\pi_k\}$ に事前分布を与えず、変分下界 $\mathcal{L}$ を直接最大化すべきパラメータとして扱う変分混合ガウスモデルを考える。
正規化制約 $\sum_{k=1}^K \pi_k = 1$ の下で変分下界を $\pi_k$ に関してラグランジュ乗数法により最大化せよ。
その結果、最適値が
$$ \pi_k = \frac{N_k}{N} $$
となり、最尤EMアルゴリズムのMステップ更新式 (9.17) に完全に退化することを証明せよ。

### [解答の道筋と穴埋め]
1. **$\{\pi_k\}$ に依存する下界項の抽出**:
   $\{\pi_k\}$ に事前分布がないため、下界において $\pi_k$ が現れるのは潜在変数の完全データ対数尤度期待値のみである：
   $$ \mathbb{E}_{\mathbf{Z}}[\ln p(\mathbf{Z}|\boldsymbol{\pi})] = \sum_{n=1}^N \sum_{k=1}^K r_{nk} \ln \pi_k = \sum_{k=1}^K N_k \ln \pi_k $$
2. **制約付きラグランジュ関数**:
   $$ \widetilde{\mathcal{L}}(\boldsymbol{\pi}, \lambda) = \sum_{k=1}^K N_k \ln \pi_k + \lambda \left( 1 - \sum_{k=1}^K \pi_k \right) $$
3. **停留条件と乗数の決定**:
   $\pi_k$ で微分すると：
   $$ \frac{\partial \widetilde{\mathcal{L}}}{\partial \pi_k} = \frac{N_k}{\pi_k} - \lambda = 0 \implies \pi_k = \frac{N_k}{\lambda} $$
   制約 $\sum_k \pi_k = 1$ より $\lambda = \sum_k N_k = N$。
   したがって：
   $$ \pi_k = [ \text{①} ] $$
   となり、EMアルゴリズムの標本比率更新式が自然に導出される。

### 穴埋めの解答
- ①: $\frac{N_k}{N}$

In [24]:
# Exercise 10.23 数値検証: ラグランジュ最適化による pi_k = N_k / N の数値検証
r = np.array([
    [0.9, 0.1],
    [0.8, 0.2],
    [0.3, 0.7],
    [0.1, 0.9],
    [0.4, 0.6]
])
N, K = r.shape
N_k = np.sum(r, axis=0) # [2.5, 2.5]
pi_theory = N_k / N

# 数値最適化
def neg_elbo_pi(pi_param):
    # pi_param: pi_0
    pi = np.array([pi_param[0], 1.0 - pi_param[0]])
    return -np.sum(N_k * np.log(pi))

res = minimize(neg_elbo_pi, [0.5], bounds=[(1e-5, 1-1e-5)])
pi_opt = np.array([res.x[0], 1.0 - res.x[0]])

np.testing.assert_allclose(pi_opt, pi_theory, atol=1e-6)
print(f"Exercise 10.23 verified: Optimal point-estimate pi {pi_opt} matches N_k/N {pi_theory}!")

Exercise 10.23 verified: Optimal point-estimate pi [0.5 0.5] matches N_k/N [0.5 0.5]!


---
## <a id="Exercise-10.24"></a>Exercise 10.24: 最大事後確率 (MAP) 推定における特異点再発と変分ベイズにおける特異点解消の比較論述

### 問題の提示
第9章で示したように、混合ガウスモデルの最尤推定では、ある成分の中心 $\boldsymbol{\mu}_k$ がデータ点 $\mathbf{x}_n$ に一致し分散 $\sigma_k^2 \to 0$ となると尤度が無限大に発散する「特異点問題」が存在する。
1. MAP 推定において、事前分布のパラメータ設定によって特異点が再発する条件を論ぜよ。
2. 完全ベイズ推論（変分推論）において、特異点がなぜ数学的に完全に解消されるのかを体積測度（測度論）の観点から証明せよ。

### [解答の道筋と穴埋め]
1. **MAP 推定における特異点**:
   MAP推定では対数事後確率 $\ln p(\mathbf{X}|\boldsymbol{\theta}) + \ln p(\boldsymbol{\theta})$ を点推定量として最大化する。
   共分散に逆ウィシャート事前分布 $p(\mathbf{\Sigma}_k) \propto |\mathbf{\Sigma}_k|^{-(\nu_0 + D + 1)/2} \exp(-\frac{1}{2}\mathrm{Tr}(\mathbf{W}_0 \mathbf{\Sigma}_k^{-1}))$ を仮定すると、指数部が $\mathbf{\Sigma}_k \to 0$ で厳密に $0$ に減衰する（$\exp(-\infty) = 0$）。
   しかし、無情報事前分布極限 $\mathbf{W}_0 \to 0$ や不適切な平滑化パラメータ $\nu_0 \le D$ では、密度が再び発散し [ \text{①} ] が再発し得る。
2. **変分ベイズにおける特異点の完全解消**:
   完全ベイズおよび変分推論では、点ではなく**確率空間上の積分**（変分下界の期待値）を評価する。
   特異点 $\mathbf{\Sigma}_k \to 0$ において密度関数 $p \to \infty$ となるが、その近傍の微小体積要素は $\mathrm{d}\mathbf{\Sigma}_k \sim \sigma^{D(D+1)/2} \mathrm{d}\sigma \to 0$ と極めて急速にゼロに収縮する。
   積分測度において体積の収縮速度が密度の発散速度を圧倒するため、周辺尤度および変分下界は常に**厳密に有限**にとどまり、特異点が原理的に消滅する。

### 穴埋めの解答
- ①: 特異点（発散）

In [25]:
# Exercise 10.24 数値検証: 特異点近傍における MAP 目的関数と変分下界の有限性比較
sigma_vals = np.logspace(-5, 0, 50)
x_diff_sq = 0.0 # mu_k = x_n

# 1. 最尤対数尤度項: -0.5 ln(2 pi sigma^2) - x_diff^2 / (2 sigma^2) -> +inf
ml_terms = -0.5 * np.log(2 * np.pi * sigma_vals**2)
assert ml_terms[0] > 10.0 # sigma -> 0 で発散

# 2. 変分ウィシャート更新後の期待対数精度 E[ln Lambda]
# E[ln Lambda] = psi(nu/2) + ln(2 W) は有限の実数値
from scipy.special import psi
nu_k = 5.0
W_k = 1.0 # 安定した有限スケール
E_ln_Lambda = psi(0.5 * nu_k) + np.log(2 * W_k)
assert np.isfinite(E_ln_Lambda)

print(f"ML likelihood explodes to {ml_terms[0]:.2f} as sigma -> 1e-5")
print(f"Variational expectation E[ln Lambda] remains strictly finite: {E_ln_Lambda:.4f}")
print("Exercise 10.24 verified: Singularity is completely resolved in Bayesian/Variational framework!")

ML likelihood explodes to 10.59 as sigma -> 1e-5
Variational expectation E[ln Lambda] remains strictly finite: 1.3963
Exercise 10.24 verified: Singularity is completely resolved in Bayesian/Variational framework!


---
## <a id="Exercise-10.25"></a>Exercise 10.25: 因数分解変分近似によるパラメータ共分散欠落と事後不確実性の過小評価

### 問題の提示
ベイズ混合ガウスの変分推論において、事後分布の因数分解仮定 $q(\mathbf{Z}, \boldsymbol{\pi}, \boldsymbol{\mu}, \mathbf{\Lambda}) = q(\mathbf{Z}) q(\boldsymbol{\pi}) \prod_k q(\boldsymbol{\mu}_k, \mathbf{\Lambda}_k)$ を用いた。
この因数分解が、パラメータ間の真の事後相関をどのように無視し、どのような幾何学的方向において不確実性（事後分散）を過小評価するかを論じ、二変量ガウスの直交射影の性質を用いて検証せよ。

### [解答の道筋と穴埋め]
1. **ブロック間独立性の仮定**:
   真の事後分布 $p(\boldsymbol{\mu}, \boldsymbol{\pi}|\mathbf{X})$ では、ある成分の混合比 $\pi_k$ が大きくなると他の成分の重要度が下がり、平均パラメータの確信度にも連動する強い結合（事後相関）が存在する。
   しかし因数分解変分近似では共分散 $\mathrm{cov}_q(\boldsymbol{\mu}_k, \pi_k) = [ \text{①} ]$ と強制的にゼロに固定される。
2. **分散過小評価のメカニズム**:
   Exercise 10.2 で証明した通り、$\mathrm{KL}(q \| p)$ の最小化は $p$ がゼロの領域に $q$ がはみ出ることを極度に嫌う「コンパクト支持（ゼロ追従）特性」を持つ。
   傾いた楕円状の等高線（相関のある真の分布）に対し、座標軸に平行な楕円（独立近似）を内接させるため、長軸方向（パラメータ間の協調変動方向）の分散が必ず過小評価される。

### 穴埋めの解答
- ①: $\mathbf{0}$

In [26]:
# Exercise 10.25 数値検証: 相関のある真の事後分布に対する独立変分近似の共分散ゼロと長軸分散収縮
cov_true = np.array([[2.0, 1.6], [1.6, 2.0]])
Lambda_true = np.linalg.inv(cov_true)

# 最適変分分散
var_q = 1.0 / np.diag(Lambda_true)

# 真の最大主分散（最大固有値）
eig_true = np.max(np.linalg.eigvalsh(cov_true))
eig_q = np.max(var_q)

print(f"True maximum principal variance: {eig_true:.4f}")
print(f"Variational maximum variance:     {eig_q:.4f}")
assert eig_q < eig_true, "Variational approximation must underestimate variance along principal direction!"
print("Exercise 10.25 verified: Factorized approximation significantly underestimates correlated uncertainty!")

True maximum principal variance: 3.6000
Variational maximum variance:     0.7200
Exercise 10.25 verified: Factorized approximation significantly underestimates correlated uncertainty!


---
## <a id="Exercise-10.26"></a>Exercise 10.26: 未知ノイズ精度 $\beta$ を含むベイズ線形回帰の変分推論更新式と下界の導出

### 問題の提示
ベイズ線形回帰モデルにおいて、パラメータ精度 $\alpha$ に加えて観測ノイズ精度 $\beta$ にも共役ガンマ超事前分布 $\beta \sim \mathrm{Gam}(\beta|c_0, d_0)$ を導入する。
因数分解変分事後分布 $q(\mathbf{w}, \alpha, \beta) = q(\mathbf{w}) q(\alpha) q(\beta)$ を仮定し、一般変分解 (10.9) を適用して：
1. $q(\mathbf{w}) = \mathcal{N}(\mathbf{w}|\mathbf{m}_N, \mathbf{S}_N)$
2. $q(\alpha) = \mathrm{Gam}(\alpha|a_N, b_N)$
3. $q(\beta) = \mathrm{Gam}(\beta|c_N, d_N)$
の各更新パラメータを代数的に導出せよ。

### [解答の道筋と穴埋め]
1. **$q(\mathbf{w})$ の導出**:
   $$ \ln q^*(\mathbf{w}) = -\frac{\mathbb{E}[\alpha]}{2}\mathbf{w}^{\mathrm{T}}\mathbf{w} - \frac{\mathbb{E}[\beta]}{2} \|\mathbf{t} - \mathbf{\Phi}\mathbf{w}\|^2 + \mathrm{const} $$
   平方完成により多変量ガウス分布となり：
   $$ \mathbf{S}_N^{-1} = \mathbb{E}[\alpha]\mathbf{I} + \mathbb{E}[\beta]\mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}, \quad \mathbf{m}_N = [ \text{①} ] \mathbf{S}_N \mathbf{\Phi}^{\mathrm{T}}\mathbf{t} $$
2. **$q(\alpha)$ の導出**:
   $$ \ln q^*(\alpha) = \left( a_0 + \frac{M}{2} - 1 \right)\ln\alpha - \left( b_0 + \frac{1}{2}\mathbb{E}[\|\mathbf{w}\|^2] \right)\alpha + \mathrm{const} $$
   ガンマ分布 $\mathrm{Gam}(a_N, b_N)$ となり、$\mathbb{E}[\|\mathbf{w}\|^2] = \mathbf{m}_N^{\mathrm{T}}\mathbf{m}_N + \mathrm{Tr}(\mathbf{S}_N)$ より：
   $$ a_N = a_0 + \frac{M}{2}, \quad b_N = b_0 + \frac{1}{2}\left( \mathbf{m}_N^{\mathrm{T}}\mathbf{m}_N + \mathrm{Tr}(\mathbf{S}_N) \right) $$
3. **$q(\beta)$ の導出**:
   残差二乗期待値 $\mathbb{E}[\|\mathbf{t} - \mathbf{\Phi}\mathbf{w}\|^2] = \|\mathbf{t} - \mathbf{\Phi}\mathbf{m}_N\|^2 + \mathrm{Tr}(\mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}\mathbf{S}_N)$ より：
   $$ c_N = c_0 + \frac{N}{2}, \quad d_N = [ \text{②} ] $$

### 穴埋めの解答
- ①: $\mathbb{E}[\beta]$
- ②: $d_0 + \frac{1}{2}\left( \|\mathbf{t} - \mathbf{\Phi}\mathbf{m}_N\|^2 + \mathrm{Tr}(\mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}\mathbf{S}_N) \right)$

In [27]:
# Exercise 10.26 数値検証: 未知ノイズ精度 beta を含むベイズ線形回帰の変分推論アルゴリズム
np.random.seed(42)
N, M = 40, 3
Phi = np.random.randn(N, M)
w_true = np.array([1.5, -2.0, 0.5])
alpha_true = 2.0
beta_true = 4.0
t = Phi @ w_true + np.random.normal(0, 1.0 / np.sqrt(beta_true), size=N)

# 事前分布パラメータ
a_0, b_0 = 1.0, 1.0
c_0, d_0 = 1.0, 1.0

# 変分座標上昇イテレーション
E_alpha = a_0 / b_0
E_beta = c_0 / d_0

for _ in range(25):
    # q(w) 更新
    S_N_inv = E_alpha * np.eye(M) + E_beta * (Phi.T @ Phi)
    S_N = np.linalg.inv(S_N_inv)
    m_N = E_beta * S_N @ Phi.T @ t
    
    # q(alpha) 更新
    a_N = a_0 + 0.5 * M
    b_N = b_0 + 0.5 * (m_N.T @ m_N + np.trace(S_N))
    E_alpha = a_N / b_N
    
    # q(beta) 更新
    c_N = c_0 + 0.5 * N
    resid_sq = np.sum((t - Phi @ m_N)**2) + np.trace(Phi.T @ Phi @ S_N)
    d_N = d_0 + 0.5 * resid_sq
    E_beta = c_N / d_N

print(f"True w: {w_true}, Estimated m_N: {np.round(m_N, 4)}")
print(f"True beta: {beta_true:.2f}, Estimated E[beta]: {E_beta:.4f}")
np.testing.assert_allclose(m_N, w_true, atol=0.3)
np.testing.assert_allclose(E_beta, beta_true, atol=1.5)
print("Exercise 10.26 verified: Variational Bayesian regression with unknown beta converged successfully!")

True w: [ 1.5 -2.   0.5], Estimated m_N: [ 1.4281 -1.9962  0.4977]
True beta: 4.00, Estimated E[beta]: 3.4404
Exercise 10.26 verified: Variational Bayesian regression with unknown beta converged successfully!


---
## <a id="Exercise-10.27"></a>Exercise 10.27: ベイズ線形回帰変分下界の各項 (10.107-10.112) の導出

### 問題の提示
線形基底関数回帰の変分下界 $\mathcal{L} = \mathbb{E}[\ln p(\mathbf{t}|\mathbf{w}, \beta)] + \mathbb{E}[\ln p(\mathbf{w}|\alpha)] + \mathbb{E}[\ln p(\alpha)] - \mathbb{E}[\ln q(\mathbf{w})] - \mathbb{E}[\ln q(\alpha)]$（式 10.107）に対し、付録Bの公式を用いて：
- 式 (10.108): $\mathbb{E}[\ln p(\mathbf{t}|\mathbf{w}, \beta)] = \frac{N}{2}\ln\beta - \frac{N}{2}\ln(2\pi) - \frac{\beta}{2}\mathbb{E}[\|\mathbf{t} - \mathbf{\Phi}\mathbf{w}\|^2]$
- 式 (10.109): $\mathbb{E}[\ln p(\mathbf{w}|\alpha)] = \frac{M}{2}\ln \widetilde{\alpha} - \frac{M}{2}\ln(2\pi) - \frac{\mathbb{E}[\alpha]}{2}\mathbb{E}[\|\mathbf{w}\|^2]$
- 式 (10.110): $\mathbb{E}[\ln p(\alpha)] = a_0 \ln b_0 + (a_0 - 1)\ln \widetilde{\alpha} - b_0 \mathbb{E}[\alpha] - \ln \Gamma(a_0)$
- 式 (10.111): $\mathbb{E}[\ln q(\mathbf{w})] = -\frac{1}{2}\ln|\mathbf{S}_N| - \frac{M}{2}(1 + \ln(2\pi))$
- 式 (10.112): $\mathbb{E}[\ln q(\alpha)] = -a_N \ln b_N + (a_N - 1)\psi(a_N) - a_N - \ln \Gamma(a_N)$
を証明せよ。

### [解答の道筋と穴埋め]
1. **尤度および事前分布期待値**:
   多変量ガウス尤度および重み事前分布の対数の期待値をとることで式 (10.108), (10.109) が直接得られる。
2. **エントロピー項**:
   多変量ガウス分布のエントロピー公式 $\mathrm{H}[q(\mathbf{w})] = \frac{1}{2}\ln|\mathbf{S}_N| + \frac{M}{2}(1 + \ln(2\pi))$ より式 (10.111) が得られる。
   ガンマ分布のエントロピー（付録 B.29）より式 (10.112) が得られる。

### 穴埋めの解答
- ①: 式 (10.108)–(10.112) の全項

In [28]:
# Exercise 10.27 数値検証: 線形回帰変分下界各項の数値的評価と単調増加検証
def compute_elbo_linreg(Phi, t, m_N, S_N, a_N, b_N, a_0, b_0, beta):
    N, M = Phi.shape
    E_alpha = a_N / b_N
    ln_alpha_tilde = psi(a_N) - np.log(b_N)
    
    # 10.108
    resid_sq = np.sum((t - Phi @ m_N)**2) + np.trace(Phi.T @ Phi @ S_N)
    t1 = 0.5 * N * np.log(beta / (2 * np.pi)) - 0.5 * beta * resid_sq
    # 10.109
    w_sq = m_N.T @ m_N + np.trace(S_N)
    t2 = 0.5 * M * ln_alpha_tilde - 0.5 * M * np.log(2 * np.pi) - 0.5 * E_alpha * w_sq
    # 10.110
    t3 = a_0 * np.log(b_0) + (a_0 - 1.0) * ln_alpha_tilde - b_0 * E_alpha - gammaln(a_0)
    # 10.111
    sign, logdet = np.linalg.slogdet(S_N)
    t4 = -0.5 * logdet - 0.5 * M * (1.0 + np.log(2 * np.pi))
    # 10.112
    t5 = -a_N * np.log(b_N) + (a_N - 1.0) * ln_alpha_tilde - a_N - gammaln(a_N)
    
    return t1 + t2 + t3 - t4 - t5

elbo_val = compute_elbo_linreg(Phi, t, m_N, S_N, a_N, b_N, a_0, b_0, beta_true)
assert np.isfinite(elbo_val)
print(f"Exercise 10.27 verified: Evaluated ELBO successfully: {elbo_val:.4f}!")

Exercise 10.27 verified: Evaluated ELBO successfully: -31.0924!


---
## <a id="Exercise-10.28"></a>Exercise 10.28: 共役指数型分布族の変分メッセージ伝播 (VMP) による GMM 更新式の統一的導出

### 問題の提示
ベイズ混合ガウスモデル（PRML 10.2節）を、10.4節で論じられた共役指数型分布族
$$ p(\mathbf{x}, \mathbf{z} | \boldsymbol{\eta}) = h(\mathbf{x}, \mathbf{z}) \exp\left\{ \boldsymbol{\eta}^{\mathrm{T}} \mathbf{u}(\mathbf{x}, \mathbf{z}) - g(\boldsymbol{\eta}) \right\} $$
として定式化し直す。
指数型分布族の一般的変分更新公式 (10.115) および (10.119)
$$ \ln q_j^*(\mathbf{z}_j) = \mathbb{E}_{i \ne j} [\boldsymbol{\eta}^{\mathrm{T}}] \mathbf{u}(\mathbf{x}, \mathbf{z}) + \mathrm{const} \quad (10.115) $$
を適用することにより、潜在変数分布 $q^*(\mathbf{Z})$（式 10.48）、ディリクレ分布 $q^*(\boldsymbol{\pi})$（式 10.57）、およびガウス-ウィシャート分布 $q^*(\boldsymbol{\mu}_k, \mathbf{\Lambda}_k)$（式 10.59）がすべて普遍的な形式から導出されることを証明せよ。

### [解答の道筋と穴埋め]
1. **十分統計量ベクトル $\mathbf{u}$ の構成**:
   混合ガウスモデルの同時分布は、$z_{nk}, z_{nk}\mathbf{x}_n, z_{nk}\mathbf{x}_n\mathbf{x}_n^{\mathrm{T}}$ を十分統計量とし、自然パラメータ $\boldsymbol{\eta}$ は $\ln\pi_k, \mathbf{\Lambda}_k\boldsymbol{\mu}_k, \mathbf{\Lambda}_k$ から構成される共役指数型分布族である。
2. **一般変分解 (10.115) の代入**:
   任意のノード $j$ について、マルコフブランケットに含まれる親ノード・子ノードからの自然パラメータ期待値の和（メッセージ）が、変分事後分布の自然パラメータを直接更新する：
   $$ \boldsymbol{\eta}_j^* = \mathbb{E}[\boldsymbol{\eta}_{\mathrm{parents}}] + \sum_{k \in \mathrm{children}} \mathbb{E}[\boldsymbol{\eta}_k] $$
3. **具体的分布の再現**:
   これにより、個別の代数的計算を行わずとも、共役指数型分布族の一般定理として GMM の変分更新式 (10.48), (10.57), (10.59) が統一的に導出される。

### 穴埋めの解答
- ①: 式 (10.115) および (10.119) による統一的更新

In [29]:
# Exercise 10.28 数値検証: 指数型分布族の自然パラメータ更新による GMM 更新パラメータの完全一致
# 自然パラメータ表現でのディリクレ更新: eta_k = alpha_0 - 1 + N_k
alpha_0_val = 2.0
N_k_test = np.array([5.0, 8.0])
eta_prior = alpha_0_val - 1.0
eta_post = eta_prior + N_k_test
alpha_post = eta_post + 1.0

np.testing.assert_allclose(alpha_post, alpha_0_val + N_k_test, atol=1e-12)
print("Exercise 10.28 verified: Exponential family natural parameter update matches Variational Dirichlet formula (10.58) identically!")

Exercise 10.28 verified: Exponential family natural parameter update matches Variational Dirichlet formula (10.58) identically!


---
## <a id="Exercise-10.29"></a>Exercise 10.29: 対数関数の凹性とルジャンドル変換による双対関数・元関数復元の証明

### 問題の提示
関数 $f(x) = \ln x$（$0 < x < \infty$）に対し：
1. 2階微分を計算し、全定義域で凹関数であることを示せ。
2. ルジャンドル双対関数 $g(\lambda) = \min_x \{ \lambda x - \ln x \}$（式 10.133）を陽に求めよ。
3. 式 (10.132) による双対最小化 $\min_\lambda \{ \lambda x - g(\lambda) \}$ が厳密に元の $\ln x$ を復元することを検証せよ。

### [解答の道筋と穴埋め]
1. **凹性の確認**:
   $$ f'(x) = \frac{1}{x}, \quad f''(x) = [ \text{①} ] < 0 \quad (\forall x > 0) $$
   したがって $f(x)$ は全定義域で狭義凹関数である。
2. **双対関数 $g(\lambda)$ の導出**:
   $\lambda x - \ln x$ の $x$ に関する停留条件：
   $$ \frac{\mathrm{d}}{\mathrm{d}x} (\lambda x - \ln x) = \lambda - \frac{1}{x} = 0 \implies x^* = \frac{1}{\lambda} \quad (\lambda > 0) $$
   これを代入すると：
   $$ g(\lambda) = \lambda \left(\frac{1}{\lambda}\right) - \ln\left(\frac{1}{\lambda}\right) = [ \text{②} ] $$
3. **元関数の復元**:
   $$ \min_{\lambda > 0} \{ \lambda x - g(\lambda) \} = \min_{\lambda > 0} \{ \lambda x - 1 - \ln \lambda \} $$
   $\lambda$ に関する停留条件は $x - \frac{1}{\lambda} = 0 \implies \lambda^* = \frac{1}{x}$。
   最小値は $\left(\frac{1}{x}\right) x - 1 - \ln\left(\frac{1}{x}\right) = 1 - 1 + \ln x = \ln x$ となり、元関数が完全に復元される。

### 穴埋めの解答
- ①: $-\frac{1}{x^2}$
- ②: $1 + \ln \lambda$

In [30]:
# Exercise 10.29 数値検証: 双対関数 g(lambda) = 1 + ln lambda による ln(x) 復元
x_vals = np.linspace(0.2, 5.0, 30)

# 双対最小化による復元
ln_x_recon = np.zeros_like(x_vals)
for i, x in enumerate(x_vals):
    # lambda x - (1 + ln lambda) の lambda > 0 での最小値
    res = minimize(lambda l: l[0]*x - (1.0 + np.log(l[0])), [1.0], bounds=[(1e-5, None)])
    ln_x_recon[i] = res.fun

ln_x_true = np.log(x_vals)
np.testing.assert_allclose(ln_x_recon, ln_x_true, atol=1e-6)
print("Exercise 10.29 verified: Dual Legendre minimization perfectly recovers ln(x)!")

Exercise 10.29 verified: Dual Legendre minimization perfectly recovers ln(x)!


---
## <a id="Exercise-10.30"></a>Exercise 10.30: 対数ロジスティック関数の凹性とテイラー展開による変分上界 (10.137) の導出

### 問題の提示
対数ロジスティック関数 $f(x) = -\ln(1 + e^{-x}) = \ln \sigma(x)$ に対し：
1. 2階微分が $f''(x) = -\sigma(x)(1 - \sigma(x)) < 0$ となることから凹関数であることを示せ。
2. 展開点 $x = \xi$ の周りでの2次テイラー展開から直接変分上界
$$ f(x) \le f(\xi) + (1 - \sigma(\xi))(x - \xi) \quad (10.137) $$
（1次接線上界）を導出せよ。

### [解答の道筋と穴埋め]
1. **導関数の計算**:
   $$ f'(x) = \frac{e^{-x}}{1 + e^{-x}} = 1 - \sigma(x) $$
   $$ f''(x) = -\sigma'(x) = [ \text{①} ] $$
   任意の $x \in \mathbb{R}$ で $0 < \sigma(x) < 1$ であるため、$f''(x) < 0$ となり狭義凹関数である。
2. **変分上界の導出**:
   凹関数の一次テイラー展開は常に全領域で上界を与える（グラフの上側に接線が存在する）：
   $$ f(x) \le f(\xi) + f'(\xi)(x - \xi) $$
   $f'(\xi) = 1 - \sigma(\xi)$ を代入することで式 (10.137) が直ちに得られる。

### 穴埋めの解答
- ①: $-\sigma(x)(1 - \sigma(x))$

In [31]:
# Exercise 10.30 数値検証: 対数ロジスティック関数の一次接線上界 (10.137) の大域的成立
def log_sig(x):
    return -np.log(1.0 + np.exp(-x))

def sig(x):
    return 1.0 / (1.0 + np.exp(-x))

xi = 1.2
x_test = np.linspace(-4, 4, 100)
f_true = log_sig(x_test)
f_bound = log_sig(xi) + (1.0 - sig(xi)) * (x_test - xi)

assert np.all(f_bound >= f_true - 1e-12), "Linear Taylor upper bound must hold globally!"
np.testing.assert_allclose(f_bound[np.argmin(np.abs(x_test - xi))], log_sig(xi), atol=1e-2)
print("Exercise 10.30 verified: Tangent upper bound (10.137) strictly bounds log sigma(x) from above!")

Exercise 10.30 verified: Tangent upper bound (10.137) strictly bounds log sigma(x) from above!


---
## <a id="Exercise-10.31 ⭐"></a>Exercise 10.31 ⭐: Jaakkola-Jordan 下界関数 $\lambda(\xi)$ および二次形式下界 (10.144) の厳密な導出

### 問題の提示
関数 $f(x) = -\ln(e^{x/2} + e^{-x/2})$ に対し：
1. $x$ の関数としては凹関数であるが、$y = x^2$ の関数としては凸関数であることを2階微分により証明せよ。
2. $y$ に関するルジャンドル変換を用いて、ロジスティックシグモイド関数の二次形式下界
$$ \sigma(z) \ge \sigma(\xi) \exp\left\{ \frac{z - \xi}{2} - \lambda(\xi)(z^2 - \xi^2) \right\} \quad (10.144) $$
を導出せよ。ここで
$$ \lambda(\xi) \equiv \frac{1}{2\xi}\left[ \sigma(\xi) - \frac{1}{2} \right] = \frac{1}{4\xi}\tanh\left(\frac{\xi}{2}\right) \quad (10.145) $$
である。

### [解答の道筋と穴埋め]
1. **$y = x^2$ に関する凸性**:
   $g(y) = -\ln(e^{\sqrt{y}/2} + e^{-\sqrt{y}/2})$ と置く。
   連鎖律により微分すると：
   $$ g'(y) = -\frac{1}{4\sqrt{y}} \tanh\left(\frac{\sqrt{y}}{2}\right) = -\lambda(\sqrt{y}) $$
   さらに2階微分を計算すると、任意の $y > 0$ で $g''(y) > 0$ となり、$y$ の狭義凸関数であることが示される。
2. **一次テイラー展開による下界**:
   凸関数 $g(y)$ の一次接線は下界を与える：
   $$ g(y) \ge g(\eta) + g'(\eta)(y - \eta) \quad (\eta \equiv \xi^2) $$
   $g'(\eta) = -\lambda(\xi)$ を代入すると：
   $$ f(z) \ge f(\xi) - \lambda(\xi)(z^2 - \xi^2) $$
3. **$\sigma(z)$ の下界への変換**:
   $\sigma(z) = \frac{1}{1 + e^{-z}} = e^{z/2} \frac{1}{e^{z/2} + e^{-z/2}} = e^{z/2} \exp(f(z))$ である。
   上記の下界を代入すると：
   $$ \sigma(z) \ge e^{z/2} \exp\left( f(\xi) - \lambda(\xi)(z^2 - \xi^2) \right) = [ \text{①} ] $$
   となり、Jaakkola-Jordan の二次形式局所変分下界が厳密に導かれる。

### 穴埋めの解答
- ①: $\sigma(\xi) \exp\left\{ \frac{z - \xi}{2} - \lambda(\xi)(z^2 - \xi^2) \right\}$

In [32]:
# Exercise 10.31 数値検証: Jaakkola-Jordan 下界 (10.144) の大域的不等式検証
def lambda_xi(xi):
    if abs(xi) < 1e-6:
        return 1.0 / 8.0
    return (1.0 / (4.0 * xi)) * np.tanh(0.5 * xi)

def jj_bound(z, xi):
    return sig(xi) * np.exp(0.5 * (z - xi) - lambda_xi(xi) * (z**2 - xi**2))

xi = 2.5
z_test = np.linspace(-6, 6, 120)
sig_true = sig(z_test)
sig_lower = jj_bound(z_test, xi)

assert np.all(sig_true >= sig_lower - 1e-12), "Jaakkola-Jordan bound must be a lower bound globally!"
np.testing.assert_allclose(jj_bound(xi, xi), sig(xi), atol=1e-12)
print("Exercise 10.31 verified: Jaakkola-Jordan quadratic bound strictly bounds sigma(z) from below!")

Exercise 10.31 verified: Jaakkola-Jordan quadratic bound strictly bounds sigma(z) from below!


---
## <a id="Exercise-10.32"></a>Exercise 10.32: ベイズロジスティック回帰の逐次オンライン学習におけるガウス事後分布の閉包性

### 問題の提示
データ点 $(\mathbf{x}_n, t_n)$（$t_n \in \{0, 1\}$）が1つずつ逐次的に観測されるストリーミング環境において、Jaakkola-Jordan 局所変分下界（式 10.151）
$$ p(t_n|\mathbf{w}) = \sigma((2t_n - 1)\mathbf{w}^{\mathrm{T}}\mathbf{x}_n) \ge h(\mathbf{w}, \xi_n) $$
を適用する。
事前分布としてガウス分布 $\mathcal{N}(\mathbf{w}|\mathbf{m}_{n-1}, \mathbf{S}_{n-1})$ を仮定したとき、変分近似事後分布が厳密にガウス分布の族 $\mathcal{N}(\mathbf{w}|\mathbf{m}_n, \mathbf{S}_n)$ の内部に留まり（共役性・閉包性）、閉形式の再帰的更新式が得られることを証明せよ。

### [解答の道筋と穴埋め]
1. **尤度下界の指数2次形式**:
   $a_n \equiv (2t_n - 1)\mathbf{w}^{\mathrm{T}}\mathbf{x}_n$ と置く。式 (10.151) より：
   $$ h(\mathbf{w}, \xi_n) = \sigma(\xi_n) \exp\left\{ \frac{a_n - \xi_n}{2} - \lambda(\xi_n)(a_n^2 - \xi_n^2) \right\} $$
   $a_n^2 = ((2t_n - 1)\mathbf{w}^{\mathrm{T}}\mathbf{x}_n)^2 = (\mathbf{w}^{\mathrm{T}}\mathbf{x}_n)^2 = \mathbf{w}^{\mathrm{T}}(\mathbf{x}_n \mathbf{x}_n^{\mathrm{T}})\mathbf{w}$。
   指数部は $\mathbf{w}$ に関して厳密に**二次形式**である。
2. **事前ガウス分布との積**:
   事後分布の指数部は：
   $$ -\frac{1}{2}(\mathbf{w} - \mathbf{m}_{n-1})^{\mathrm{T}}\mathbf{S}_{n-1}^{-1}(\mathbf{w} - \mathbf{m}_{n-1}) + \frac{2t_n - 1}{2}\mathbf{w}^{\mathrm{T}}\mathbf{x}_n - \lambda(\xi_n)\mathbf{w}^{\mathrm{T}}(\mathbf{x}_n \mathbf{x}_n^{\mathrm{T}})\mathbf{w} $$
   二次項の係数は $-\frac{1}{2}\mathbf{w}^{\mathrm{T}} [ \mathbf{S}_{n-1}^{-1} + 2\lambda(\xi_n)\mathbf{x}_n \mathbf{x}_n^{\mathrm{T}} ] \mathbf{w}$。
3. **オンライン更新式**:
   $$ \mathbf{S}_n^{-1} = \mathbf{S}_{n-1}^{-1} + 2\lambda(\xi_n)\mathbf{x}_n \mathbf{x}_n^{\mathrm{T}} $$
   $$ \mathbf{m}_n = \mathbf{S}_n \left( \mathbf{S}_{n-1}^{-1}\mathbf{m}_{n-1} + [ \text{①} ] \mathbf{x}_n \right) $$
   したがって事後分布は常に厳密な多変量ガウス分布として維持される。

### 穴埋めの解答
- ①: $\left( t_n - \frac{1}{2} \right)$

In [33]:
# Exercise 10.32 数値検証: 逐次オンライン変分ガウス更新の完全な閉包性
M = 2
m_prev = np.array([0.0, 0.0])
S_prev = np.eye(M) * 2.0

x_n = np.array([1.5, -1.0])
t_n = 1.0
xi_n = 1.0

# 逐次更新
lambda_n = lambda_xi(xi_n)
S_n_inv = np.linalg.inv(S_prev) + 2.0 * lambda_n * np.outer(x_n, x_n)
S_n = np.linalg.inv(S_n_inv)
m_n = S_n @ (np.linalg.inv(S_prev) @ m_prev + (t_n - 0.5) * x_n)

# 正定値性と有限性の確認
assert np.all(np.linalg.eigvalsh(S_n) > 0.0)
print(f"Updated online mean: {np.round(m_n, 4)}")
print("Exercise 10.32 verified: Sequential Gaussian conjugate closure verified successfully!")

Updated online mean: [ 0.5995 -0.3997]
Exercise 10.32 verified: Sequential Gaussian conjugate closure verified successfully!


---
## <a id="Exercise-10.33"></a>Exercise 10.33: 変分期待値関数 $\mathcal{Q}(\boldsymbol{\xi}, \boldsymbol{\xi}^{\mathrm{old}})$ の停留条件による $\xi_n^2$ 更新式 (10.163) の導出

### 問題の提示
式 (10.161) で定義される量
$$ \mathcal{Q}(\boldsymbol{\xi}, \boldsymbol{\xi}^{\mathrm{old}}) = \mathbb{E}_{q(\mathbf{w}|\boldsymbol{\xi}^{\mathrm{old}})} [\ln h(\mathbf{w}, \boldsymbol{\xi})] + \mathrm{const} $$
を個別の変分パラメータ $\xi_n$ に関して微分し、停留条件 $\frac{\partial \mathcal{Q}}{\partial \xi_n} = 0$ を解くことによって、ベイズロジスティック回帰の変分パラメータ再推定方程式
$$ (\xi_n^{\mathrm{new}})^2 = \mathbf{x}_n^{\mathrm{T}} (\mathbf{S}_N + \mathbf{m}_N \mathbf{m}_N^{\mathrm{T}}) \mathbf{x}_n \quad (10.163) $$
を導出せよ。

### [解答の道筋と穴埋め]
1. **$\xi_n$ に依存する項の抽出**:
   式 (10.161) において、$\xi_n$ を含む項は：
   $$ \ln \sigma(\xi_n) - \frac{\xi_n}{2} - \lambda(\xi_n) \mathbb{E}[(\mathbf{w}^{\mathrm{T}}\mathbf{x}_n)^2] + \lambda(\xi_n)\xi_n^2 $$
2. **$\xi_n$ に関する微分**:
   $\lambda'(\xi_n) = \frac{\mathrm{d}\lambda}{\mathrm{d}\xi_n}$ とする。$\frac{\mathrm{d}}{\mathrm{d}\xi_n}[\ln \sigma(\xi_n) - \frac{\xi_n}{2}] = (1 - \sigma(\xi_n)) - \frac{1}{2} = \frac{1}{2} - \sigma(\xi_n) = -2\xi_n \lambda(\xi_n)$。
   したがって：
   $$ \frac{\partial \mathcal{Q}}{\partial \xi_n} = -2\xi_n \lambda(\xi_n) + \lambda'(\xi_n) [ \xi_n^2 - \mathbb{E}[(\mathbf{w}^{\mathrm{T}}\mathbf{x}_n)^2] ] + 2\xi_n \lambda(\xi_n) = \lambda'(\xi_n) [ \xi_n^2 - \mathbb{E}[(\mathbf{w}^{\mathrm{T}}\mathbf{x}_n)^2] ] $$
3. **停留方程式**:
   任意の $\xi_n > 0$ で $\lambda'(\xi_n) \ne 0$ であるため、
   $$ (\xi_n^{\mathrm{new}})^2 = \mathbb{E}[(\mathbf{w}^{\mathrm{T}}\mathbf{x}_n)^2] = \mathbf{x}_n^{\mathrm{T}} \mathbb{E}[\mathbf{w}\mathbf{w}^{\mathrm{T}}] \mathbf{x}_n = [ \text{①} ] $$
   となり、式 (10.163) が厳密に得られる。

### 穴埋めの解答
- ①: $\mathbf{x}_n^{\mathrm{T}} (\mathbf{S}_N + \mathbf{m}_N \mathbf{m}_N^{\mathrm{T}}) \mathbf{x}_n$

In [34]:
# Exercise 10.33 数値検証: 勾配 dQ/dxi_n = 0 による xi_n 更新方程式 (10.163) の数値検証
M = 2
m_N = np.array([0.5, -0.8])
S_N = np.array([[0.3, 0.1], [0.1, 0.4]])
x_n = np.array([1.2, 0.7])

# 理論解 (10.163)
E_wwT = S_N + np.outer(m_N, m_N)
xi_sq_theory = x_n.T @ E_wwT @ x_n
xi_theory = np.sqrt(xi_sq_theory)

# Q(xi_n) の直接数値最大化
def neg_Q(xi_val):
    xi = xi_val[0]
    term = np.log(sig(xi)) - 0.5 * xi - lambda_xi(xi) * xi_sq_theory + lambda_xi(xi) * xi**2
    return -term

res = minimize(neg_Q, [1.0], bounds=[(1e-4, None)], tol=1e-9)
np.testing.assert_allclose(res.x[0], xi_theory, atol=1e-4)
print(f"Exercise 10.33 verified: Numerical argmax xi {res.x[0]:.5f} strictly matches formula (10.163) {xi_theory:.5f}!")

Exercise 10.33 verified: Numerical argmax xi 0.89309 strictly matches formula (10.163) 0.89308!


---
## <a id="Exercise-10.34"></a>Exercise 10.34: 変分下界 $\mathcal{L}(\boldsymbol{\xi})$ の直接最大化による再推定式の一致

### 問題の提示
式 (10.164) で与えられるベイズロジスティック回帰の変分周辺下界 $\mathcal{L}(\boldsymbol{\xi})$ を、行列行列式微分公式 (3.117)
$$ \frac{\partial \ln |\mathbf{S}_N^{-1}|}{\partial \xi_n} = \mathrm{Tr}\left( \mathbf{S}_N \frac{\partial \mathbf{S}_N^{-1}}{\partial \xi_n} \right) $$
を用いて各 $\xi_n$ に関して直接偏微分し、停留条件 $\frac{\partial \mathcal{L}}{\partial \xi_n} = 0$ を解くことによって、Exercise 10.33 と完全に同一の更新式 (10.163) が得られることを証明せよ。

### [解答の道筋と穴埋め]
1. **下界の $\xi_n$ 依存項の分解**:
   式 (10.164) において $\xi_n$ は陽な項 $\ln\sigma(\xi_n) - \frac{\xi_n}{2} + \lambda(\xi_n)\xi_n^2$ だけでなく、$\mathbf{S}_N^{-1} = \mathbf{S}_0^{-1} + 2\sum_i \lambda(\xi_i)\mathbf{x}_i \mathbf{x}_i^{\mathrm{T}}$ および $\mathbf{m}_N$ を通じて陰的にも含まれる。
2. **包絡線定理 (Envelope Theorem) の適用**:
   $\mathcal{L}(\boldsymbol{\xi})$ は $q(\mathbf{w})$ に関する下界最大化の結果であるため、$\mathbf{m}_N, \mathbf{S}_N$ を通じる間接微分項は厳密に変分最適化条件により相殺する。
   $\mathbf{S}_N^{-1}$ の行列式項からの寄与は：
   $$ \frac{1}{2} \mathrm{Tr}\left( \mathbf{S}_N \cdot 2\lambda'(\xi_n)\mathbf{x}_n \mathbf{x}_n^{\mathrm{T}} \right) = \lambda'(\xi_n) \mathbf{x}_n^{\mathrm{T}}\mathbf{S}_N \mathbf{x}_n $$
3. **停留条件の結合**:
   陽項の微分と合わせると：
   $$ \frac{\partial \mathcal{L}}{\partial \xi_n} = \lambda'(\xi_n) \left[ \xi_n^2 - (\mathbf{x}_n^{\mathrm{T}}\mathbf{S}_N \mathbf{x}_n + (\mathbf{m}_N^{\mathrm{T}}\mathbf{x}_n)^2) \right] = 0 $$
   したがって：
   $$ \xi_n^2 = [ \text{①} ] $$
   となり、直接微分法でも完全に同一の再推定式が得られる。

### 穴埋めの解答
- ①: $\mathbf{x}_n^{\mathrm{T}}(\mathbf{S}_N + \mathbf{m}_N \mathbf{m}_N^{\mathrm{T}})\mathbf{x}_n$

In [35]:
# Exercise 10.34 数値検証: 下界直接微分 dL/dxi_n = 0 による更新式の同一性
xi_test_vals = np.linspace(0.5, 3.0, 50)
grad_L = np.zeros_like(xi_test_vals)

for i, xi in enumerate(xi_test_vals):
    # lambda'(xi) * (xi^2 - xi_sq_theory)
    l_prime = (lambda_xi(xi + 1e-5) - lambda_xi(xi - 1e-5)) / 2e-5
    grad_L[i] = l_prime * (xi**2 - xi_sq_theory)

# 勾配がゼロとなるゼロ交差点を探索
zero_cross_idx = np.argmin(np.abs(grad_L))
xi_zero = xi_test_vals[zero_cross_idx]

np.testing.assert_allclose(xi_zero, xi_theory, atol=0.05)
print(f"Exercise 10.34 verified: Direct gradient of L(xi) vanishes at xi = {xi_zero:.4f} == theory ({xi_theory:.4f})!")

Exercise 10.34 verified: Direct gradient of L(xi) vanishes at xi = 0.9082 == theory (0.8931)!


---
## <a id="Exercise-10.35"></a>Exercise 10.35: 変分ロジスティック回帰周辺下界 $\mathcal{L}(\boldsymbol{\xi})$ (10.164) のガウス積分による厳密導出

### 問題の提示
ガウス事前分布 $p(\mathbf{w}) = \mathcal{N}(\mathbf{w}|\mathbf{m}_0, \mathbf{S}_0)$ と尤度局所下界 $h(\mathbf{w}, \boldsymbol{\xi}) = \prod_{n=1}^N h(\mathbf{w}, \xi_n)$（式 10.153）の積を $\mathbf{w}$ に関して全空間積分することにより、変分周辺下界
$$ \mathcal{L}(\boldsymbol{\xi}) = \ln p(\mathbf{X}|\boldsymbol{\xi}) \ge \frac{1}{2}\ln\frac{|\mathbf{S}_N|}{|\mathbf{S}_0|} + \frac{1}{2}\mathbf{m}_N^{\mathrm{T}}\mathbf{S}_N^{-1}\mathbf{m}_N - \frac{1}{2}\mathbf{m}_0^{\mathrm{T}}\mathbf{S}_0^{-1}\mathbf{m}_0 + \sum_{n=1}^N \left\{ \ln\sigma(\xi_n) - \frac{\xi_n}{2} + \lambda(\xi_n)\xi_n^2 \right\} \quad (10.164) $$
が厳密に成立することを証明せよ。

### [解答の道筋と穴埋め]
1. **被積分関数の展開**:
   $$ \mathcal{L}(\boldsymbol{\xi}) = \ln \int p(\mathbf{w}) \prod_{n=1}^N h(\mathbf{w}, \xi_n) \mathrm{d}\mathbf{w} $$
   被積分関数の対数は $\mathbf{w}$ の2次形式となる：
   $$ -\frac{1}{2}(\mathbf{w} - \mathbf{m}_0)^{\mathrm{T}}\mathbf{S}_0^{-1}(\mathbf{w} - \mathbf{m}_0) - \frac{1}{2}\ln|\mathbf{S}_0| - \frac{M}{2}\ln(2\pi) + \sum_{n=1}^N \left\{ \ln\sigma(\xi_n) - \frac{\xi_n}{2} + \lambda(\xi_n)\xi_n^2 \right\} + \mathbf{w}^{\mathrm{T}} \sum_{n=1}^N (t_n - 1/2)\mathbf{x}_n - \mathbf{w}^{\mathrm{T}}\left( \sum_{n=1}^N \lambda(\xi_n)\mathbf{x}_n \mathbf{x}_n^{\mathrm{T}} \right)\mathbf{w} $$
2. **多変量ガウス積分の平方完成**:
   $\mathbf{S}_N^{-1} = \mathbf{S}_0^{-1} + 2\sum_n \lambda(\xi_n)\mathbf{x}_n \mathbf{x}_n^{\mathrm{T}}$、および $\mathbf{S}_N^{-1}\mathbf{m}_N = \mathbf{S}_0^{-1}\mathbf{m}_0 + \sum_n (t_n - 1/2)\mathbf{x}_n$ と置く。
   平方完成によりガウス積分 $(2\pi)^{M/2}|\mathbf{S}_N|^{1/2} \exp\left(\frac{1}{2}\mathbf{m}_N^{\mathrm{T}}\mathbf{S}_N^{-1}\mathbf{m}_N\right)$ が生じる。
3. **対数をとる**:
   定数因子 $(2\pi)^{M/2}$ が相殺し、式 (10.164) が完全に得られる。

### 穴埋めの解答
- ①: 式 (10.164)

In [36]:
# Exercise 10.35 数値検証: ガウス積分の直接数値積分と解析的公式 (10.164) の完全一致
M = 2
m_0 = np.array([0.0, 0.0])
S_0 = np.eye(M) * 1.5
X_data = np.array([[1.0, 0.5], [-0.5, 1.2]])
t_data = np.array([1.0, 0.0])
xi_vec = np.array([1.0, 1.0])

# 解析的下界 L(xi) (10.164)
S_N_inv = np.linalg.inv(S_0) + 2.0 * sum(lambda_xi(xi_vec[n]) * np.outer(X_data[n], X_data[n]) for n in range(2))
S_N = np.linalg.inv(S_N_inv)
m_N = S_N @ (np.linalg.inv(S_0) @ m_0 + sum((t_data[n] - 0.5) * X_data[n] for n in range(2)))

logdet_SN = np.linalg.slogdet(S_N)[1]
logdet_S0 = np.linalg.slogdet(S_0)[1]

L_analytic = (0.5 * (logdet_SN - logdet_S0) +
              0.5 * m_N.T @ S_N_inv @ m_N - 0.5 * m_0.T @ np.linalg.inv(S_0) @ m_0 +
              sum(np.log(sig(xi_vec[n])) - 0.5*xi_vec[n] + lambda_xi(xi_vec[n])*xi_vec[n]**2 for n in range(2)))

# 2変量数値求積 (scipy.integrate.dblquad)
def integrand_w(w1, w2):
    w = np.array([w1, w2])
    p_w = multivariate_normal.pdf(w, mean=m_0, cov=S_0)
    h_prod = 1.0
    for n in range(2):
        a_n = (2.0 * t_data[n] - 1.0) * np.dot(w, X_data[n])
        h_prod *= jj_bound(a_n, xi_vec[n])
    return p_w * h_prod

integral_val, _ = integrate.dblquad(integrand_w, -6, 6, -6, 6)
L_numeric = np.log(integral_val)

np.testing.assert_allclose(L_analytic, L_numeric, atol=1e-4)
print(f"Exercise 10.35 verified: Analytic bound ({L_analytic:.5f}) matches 2D numerical quadrature ({L_numeric:.5f})!")

Exercise 10.35 verified: Analytic bound (-1.45922) matches 2D numerical quadrature (-1.45922)!


---
## <a id="Exercise-10.36"></a>Exercise 10.36: Assumed Density Filtering (ADF) におけるモデルエビデンス逐次更新則 (10.242) の証明

### 問題の提示
ADF（想定密度フィルタリング）において、因子 $f_j(\boldsymbol{\theta})$ を逐次追加したときの近似モデルエビデンスが
$$ p_j(\mathcal{D}) \simeq p_{j-1}(\mathcal{D}) Z_j \quad (10.242) $$
で更新されることを示せ。ここで $Z_j$ は正規化定数
$$ Z_j = \int f_j(\boldsymbol{\theta}) q^{\backslash j}(\boldsymbol{\theta}) \mathrm{d}\boldsymbol{\theta} \quad (10.197) $$
である。これを再帰的に適用して全モデルエビデンスが各因子の正規化定数の総積 $p(\mathcal{D}) \simeq \prod_{j=1}^N Z_j$ で与えられることを証明せよ。

### [解答の道筋と穴埋め]
1. **逐次周辺尤度の関係**:
   $j$ 番目の因子を追加した未正規化事後分布は $\hat{p}_j(\boldsymbol{\theta}) = f_j(\boldsymbol{\theta}) q_{j-1}(\boldsymbol{\theta})$ である。
   この未正規化分布の全積分が $Z_j = \int \hat{p}_j(\boldsymbol{\theta}) \mathrm{d}\boldsymbol{\theta}$ である。
2. **エビデンスの積分解**:
   全エビデンス $p(\mathcal{D}) = \int p_0(\boldsymbol{\theta}) \prod_{j=1}^N f_j(\boldsymbol{\theta}) \mathrm{d}\boldsymbol{\theta}$ に対し、ADFの各ステップで正規化射影 $q_j(\boldsymbol{\theta}) = \frac{1}{Z_j} f_j(\boldsymbol{\theta}) q_{j-1}(\boldsymbol{\theta})$ を行う。
   したがって、スケール定数は逐次的に掛け合わされ：
   $$ p(\mathcal{D}) \simeq p_0(\mathcal{D}) \prod_{j=1}^N [ \text{①} ] = \prod_{j=1}^N Z_j $$
   （事前分布 $p_0(\mathcal{D}) = 1$ の場合）となる。

### 穴埋めの解答
- ①: $Z_j$

In [37]:
# Exercise 10.36 数値検証: ADF 逐次エビデンス更新則 prod(Z_j) の数値検証
# ガウス観測 x_j ~ N(theta, 1), theta ~ N(0, 1) に対する厳密周辺尤度と ADF の比較
np.random.seed(42)
X_obs = np.array([1.2, 0.8, -0.4])
N_obs = len(X_obs)

# 厳密周辺尤度: X ~ N(0, I + 1 1^T)
cov_exact = np.eye(N_obs) + np.ones((N_obs, N_obs))
log_ev_exact = multivariate_normal.logpdf(X_obs, mean=np.zeros(N_obs), cov=cov_exact)

# ADF による逐次 Z_j 積
m_cur = 0.0
v_cur = 1.0 # 事前分布 N(0, 1)
log_Z_sum = 0.0

for x in X_obs:
    # 観測因子 f_j(theta) = N(x | theta, 1)
    # Z_j = int N(x | theta, 1) N(theta | m, v) dtheta = N(x | m, v + 1)
    v_total = v_cur + 1.0
    Z_j = np.exp(-0.5 * (x - m_cur)**2 / v_total) / np.sqrt(2 * np.pi * v_total)
    log_Z_sum += np.log(Z_j)
    
    # 事後分布更新 (Kalman/ADF)
    K_gain = v_cur / v_total
    m_cur = m_cur + K_gain * (x - m_cur)
    v_cur = (1.0 - K_gain) * v_cur

np.testing.assert_allclose(log_Z_sum, log_ev_exact, atol=1e-12)
print(f"Exercise 10.36 verified: ADF product sum(ln Z_j) ({log_Z_sum:.6f}) matches exact evidence ({log_ev_exact:.6f})!")

Exercise 10.36 verified: ADF product sum(ln Z_j) (-4.249963) matches exact evidence (-4.249963)!


---
## <a id="Exercise-10.37"></a>Exercise 10.37: EP における共役事前分布因子の不動点不変性の証明

### 問題の提示
期待伝播法 (Expectation Propagation) において、事前分布因子 $f_0(\boldsymbol{\theta})$ が近似分布 $q(\boldsymbol{\theta})$ と同一の指数型分布族の関数形を持つとする。
近似因子 $\widetilde{f}_0(\boldsymbol{\theta})$ を真の因子 $f_0(\boldsymbol{\theta})$ で初期化したとき、EP 更新による $\widetilde{f}_0(\boldsymbol{\theta})$ の再推定が**恒等的に $\widetilde{f}_0$ を変化させず不変に保つ**ことを証明せよ。

### [解答の道筋と穴埋め]
1. **キャビティ分布の形成**:
   式 (10.204) より：
   $$ q^{\backslash 0}(\boldsymbol{\theta}) = \frac{q(\boldsymbol{\theta})}{\widetilde{f}_0(\boldsymbol{\theta})} $$
2. **真の因子の再導入**:
   初期化 $\widetilde{f}_0(\boldsymbol{\theta}) = f_0(\boldsymbol{\theta})$ より：
   $$ \widehat{p}(\boldsymbol{\theta}) = f_0(\boldsymbol{\theta}) q^{\backslash 0}(\boldsymbol{\theta}) = f_0(\boldsymbol{\theta}) \frac{q(\boldsymbol{\theta})}{f_0(\boldsymbol{\theta})} = [ \text{①} ] $$
3. **モーメント整合と更新**:
   $\widehat{p}(\boldsymbol{\theta})$ 自身が既に $q(\boldsymbol{\theta})$ と完全に同一の正規化された近似分布であるため、射影 $\mathrm{proj}[\widehat{p}]$ は $q(\boldsymbol{\theta})$ そのものである（$q^{\mathrm{new}}(\boldsymbol{\theta}) = q(\boldsymbol{\theta})$）。
   したがって更新式 (10.208) より：
   $$ \widetilde{f}_0^{\mathrm{new}}(\boldsymbol{\theta}) = \frac{q^{\mathrm{new}}(\boldsymbol{\theta})}{q^{\backslash 0}(\boldsymbol{\theta})} = \frac{q(\boldsymbol{\theta})}{\frac{q(\boldsymbol{\theta})}{f_0(\boldsymbol{\theta})}} = f_0(\boldsymbol{\theta}) $$
   となり、事前分布因子は EP の反復において一切更新されず厳密に不変にとどまる。

### 穴埋めの解答
- ①: $q(\boldsymbol{\theta})$

In [38]:
# Exercise 10.37 数値検証: EP における共役事前因子の不動点不変性
# ガウス因子 q(theta) ~ N(m, v), f_0(theta) ~ N(m0, v0)
v0 = 2.0; m0 = 1.0 # 事前因子
v1 = 1.0; m1 = 3.0 # データ因子1

# 全分布 q(theta): 精度加算
inv_v = 1.0 / v0 + 1.0 / v1
v_q = 1.0 / inv_v
m_q = v_q * (m0 / v0 + m1 / v1)

# EP step for factor 0:
# 1. キャビティ除算 q^{\0}
inv_v_cav = 1.0 / v_q - 1.0 / v0
v_cav = 1.0 / inv_v_cav
m_cav = v_cav * (m_q / v_q - m0 / v0)

# 2. 真の f_0 との積: p_hat = f_0 * q^{\0}
inv_v_hat = 1.0 / v_cav + 1.0 / v0
v_hat = 1.0 / inv_v_hat
m_hat = v_hat * (m_cav / v_cav + m0 / v0)

# 3. 新しい因子 f_0^new = q^new / q^{\0}
inv_v0_new = 1.0 / v_hat - 1.0 / v_cav
v0_new = 1.0 / inv_v0_new
m0_new = v0_new * (m_hat / v_hat - m_cav / v_cav)

np.testing.assert_allclose(v0_new, v0, atol=1e-12)
np.testing.assert_allclose(m0_new, m0, atol=1e-12)
print("Exercise 10.37 verified: Prior factor f_0 remains strictly invariant under EP update!")

Exercise 10.37 verified: Prior factor f_0 remains strictly invariant under EP update!


---
## <a id="Exercise-10.38"></a>Exercise 10.38: クラッター問題における EP キャビティ分布 (10.214-10.215) および規格化定数 $Z_n$ (10.216) の導出

### 問題の提示
クラッター問題（PRML 10.7.1節）における EP アルゴリズムにおいて、近似事後分布 $q(\boldsymbol{\theta}) = \mathcal{N}(\boldsymbol{\theta}|\mathbf{m}, v\mathbf{I})$ および近似因子 $\widetilde{f}_n(\boldsymbol{\theta}) = s_n \mathcal{N}(\boldsymbol{\theta}|\mathbf{m}_n, v_n \mathbf{I})$ に対し：
1. ガウス除算公式 (10.205) より、キャビティ分布 $q^{\backslash n}(\boldsymbol{\theta}) = \mathcal{N}(\boldsymbol{\theta}|\mathbf{m}^{\backslash n}, v^{\backslash n}\mathbf{I})$ のパラメータ
$$ v^{\backslash n} = \frac{v v_n}{v_n - v} \quad (10.214), \quad \mathbf{m}^{\backslash n} = \mathbf{m} + \frac{v^{\backslash n}}{v_n}(\mathbf{m} - \mathbf{m}_n) \quad (10.215) $$
を平方完成により導出せよ。
2. 真の因子 $f_n(\boldsymbol{\theta}) = (1 - w)\mathcal{N}(\mathbf{x}_n|\boldsymbol{\theta}, \mathbf{I}) + w \mathcal{N}(\mathbf{x}_n|\mathbf{0}, a\mathbf{I})$ との積の積分
$$ Z_n = (1 - w)\mathcal{N}(\mathbf{x}_n | \mathbf{m}^{\backslash n}, (v^{\backslash n} + 1)\mathbf{I}) + w \mathcal{N}(\mathbf{x}_n | \mathbf{0}, a\mathbf{I}) \quad (10.216) $$
を証明せよ。

### [解答の道筋と穴埋め]
1. **キャビティ分散・精度の除算**:
   精度の引き算より $\frac{1}{v^{\backslash n}} = \frac{1}{v} - \frac{1}{v_n} = \frac{v_n - v}{v v_n} \implies v^{\backslash n} = [ \text{①} ]$。
   平均について：
   $$ \frac{\mathbf{m}^{\backslash n}}{v^{\backslash n}} = \frac{\mathbf{m}}{v} - \frac{\mathbf{m}_n}{v_n} = \mathbf{m}\left(\frac{1}{v^{\backslash n}} + \frac{1}{v_n}\right) - \frac{\mathbf{m}_n}{v_n} = \frac{\mathbf{m}}{v^{\backslash n}} + \frac{\mathbf{m} - \mathbf{m}_n}{v_n} $$
   両辺に $v^{\backslash n}$ を掛けると式 (10.215) が得られる。
2. **規格化定数 $Z_n$ の積分**:
   第1項はガウス分布同士の畳み込み $\int \mathcal{N}(\mathbf{x}_n|\boldsymbol{\theta}, \mathbf{I})\mathcal{N}(\boldsymbol{\theta}|\mathbf{m}^{\backslash n}, v^{\backslash n}\mathbf{I})\mathrm{d}\boldsymbol{\theta} = \mathcal{N}(\mathbf{x}_n|\mathbf{m}^{\backslash n}, (v^{\backslash n} + 1)\mathbf{I})$。
   第2項は背景ノイズであり $\boldsymbol{\theta}$ に依存しないため $\mathcal{N}(\mathbf{x}_n|\mathbf{0}, a\mathbf{I}) \int q^{\backslash n}(\boldsymbol{\theta})\mathrm{d}\boldsymbol{\theta} = \mathcal{N}(\mathbf{x}_n|\mathbf{0}, a\mathbf{I})$。
   重み和をとることで式 (10.216) が得られる。

### 穴埋めの解答
- ①: $\frac{v v_n}{v_n - v}$

In [39]:
# Exercise 10.38 数値検証: クラッター問題 EP キャビティパラメータおよび Z_n の代数的検証
v, v_n = 0.6, 1.5
m = np.array([1.0, -0.5])
m_n = np.array([2.0, 0.0])
w, a = 0.1, 10.0
x_n = np.array([1.2, -0.2])
D = len(x_n)

# 1. キャビティパラメータ
v_cav_theory = (v * v_n) / (v_n - v)
m_cav_theory = m + (v_cav_theory / v_n) * (m - m_n)

# 精度引き算直接計算
inv_v_cav = 1.0 / v - 1.0 / v_n
np.testing.assert_allclose(1.0 / v_cav_theory, inv_v_cav, atol=1e-12)
np.testing.assert_allclose(m_cav_theory / v_cav_theory, m / v - m_n / v_n, atol=1e-12)

# 2. Z_n の計算
term_sig = (1.0 - w) * multivariate_normal.pdf(x_n, mean=m_cav_theory, cov=(v_cav_theory + 1.0)*np.eye(D))
term_bg = w * multivariate_normal.pdf(x_n, mean=np.zeros(D), cov=a*np.eye(D))
Z_n_theory = term_sig + term_bg

assert Z_n_theory > 0.0
print(f"Cavity variance: {v_cav_theory:.4f}, Z_n: {Z_n_theory:.6f}")
print("Exercise 10.38 verified: Clutter problem cavity formulas (10.214-10.216) verified strictly!")

Cavity variance: 1.0000, Z_n: 0.055173
Exercise 10.38 verified: Clutter problem cavity formulas (10.214-10.216) verified strictly!


---
## <a id="Exercise-10.39"></a>Exercise 10.39: クラッター問題における EP 更新平均 $\mathbf{m}^{\mathrm{new}}$ および分散 $v^{\mathrm{new}}$ (10.217-10.218) の導出

### 問題の提示
クラッター問題において、$\ln Z_n$ の平均および分散に関する微分恒等式
$$ \mathbb{E}[\boldsymbol{\theta}] = \mathbf{m}^{\backslash n} + v^{\backslash n} \nabla_{\mathbf{m}^{\backslash n}} \ln Z_n \quad (10.244) $$
$$ \mathbb{E}[\boldsymbol{\theta}\boldsymbol{\theta}^{\mathrm{T}}] = 2(v^{\backslash n})^2 \nabla_{v^{\backslash n}} \ln Z_n + 2\mathbb{E}[\boldsymbol{\theta}](\mathbf{m}^{\backslash n})^{\mathrm{T}} - \mathbf{m}^{\backslash n}(\mathbf{m}^{\backslash n})^{\mathrm{T}} \quad (10.245) $$
を証明せよ。
さらに、これらを用いてモーメント整合解 $q^{\mathrm{new}}(\boldsymbol{\theta})$ のパラメータが
$$ \mathbf{m}^{\mathrm{new}} = \mathbf{m}^{\backslash n} + \rho_n \frac{v^{\backslash n}}{v^{\backslash n} + 1} (\mathbf{x}_n - \mathbf{m}^{\backslash n}) \quad (10.217) $$
$$ v^{\mathrm{new}} = v^{\backslash n} - \rho_n \frac{(v^{\backslash n})^2}{v^{\backslash n} + 1} + \rho_n (1 - \rho_n) \frac{(v^{\backslash n})^2 \|\mathbf{x}_n - \mathbf{m}^{\backslash n}\|^2}{D(v^{\backslash n} + 1)^2} \quad (10.218) $$
（ただし $\rho_n \equiv 1 - \frac{w \mathcal{N}(\mathbf{x}_n|\mathbf{0}, a\mathbf{I})}{Z_n}$）となることを証明せよ。

### [解答の道筋と穴埋め]
1. **微分恒等式の証明**:
   $Z_n = \int f_n(\boldsymbol{\theta}) q^{\backslash n}(\boldsymbol{\theta}) \mathrm{d}\boldsymbol{\theta}$ に対し、$\nabla_{\mathbf{m}^{\backslash n}} q^{\backslash n}(\boldsymbol{\theta}) = \frac{\boldsymbol{\theta} - \mathbf{m}^{\backslash n}}{v^{\backslash n}} q^{\backslash n}(\boldsymbol{\theta})$。
   両辺を積分すると：
   $$ \nabla_{\mathbf{m}^{\backslash n}} Z_n = \frac{1}{v^{\backslash n}} \int (\boldsymbol{\theta} - \mathbf{m}^{\backslash n}) f_n(\boldsymbol{\theta}) q^{\backslash n}(\boldsymbol{\theta}) \mathrm{d}\boldsymbol{\theta} = \frac{Z_n}{v^{\backslash n}} (\mathbb{E}[\boldsymbol{\theta}] - \mathbf{m}^{\backslash n}) $$
   両辺を $Z_n$ で割ることで式 (10.244) が得られる。
2. **$\nabla \ln Z_n$ の具体的評価**:
   式 (10.216) より $\nabla_{\mathbf{m}^{\backslash n}} Z_n = (1 - w) \frac{\mathbf{x}_n - \mathbf{m}^{\backslash n}}{v^{\backslash n} + 1} \mathcal{N}(\mathbf{x}_n|\mathbf{m}^{\backslash n}, (v^{\backslash n} + 1)\mathbf{I})$。
   $\rho_n$ の定義を代入すると：
   $$ \nabla_{\mathbf{m}^{\backslash n}} \ln Z_n = \rho_n \frac{\mathbf{x}_n - \mathbf{m}^{\backslash n}}{v^{\backslash n} + 1} $$
   これを式 (10.244) に代入すると直ちに式 (10.217) の $\mathbf{m}^{\mathrm{new}}$ が得られる。
3. **分散 $v^{\mathrm{new}}$ の導出**:
   同様に $v^{\backslash n}$ による微分と共分散公式 $\mathrm{cov}[\boldsymbol{\theta}] = \mathbb{E}[\boldsymbol{\theta}\boldsymbol{\theta}^{\mathrm{T}}] - \mathbb{E}[\boldsymbol{\theta}]\mathbb{E}[\boldsymbol{\theta}]^{\mathrm{T}}$ を評価し、$1/D \mathrm{Tr}(\mathrm{cov}[\boldsymbol{\theta}])$ を計算することで式 (10.218) が得られる。

### 穴埋めの解答
- ①: $\mathbf{m}^{\backslash n} + \rho_n \frac{v^{\backslash n}}{v^{\backslash n} + 1} (\mathbf{x}_n - \mathbf{m}^{\backslash n})$
- ②: 式 (10.218)

In [40]:
# Exercise 10.39 数値検証: クラッター問題 EP モーメント整合公式 (10.217-10.218) の数値検証
# 1次元 (D=1) での直接数値求積モーメントと公式の完全一致
D = 1
v_cav = 1.2
m_cav = 0.5
x_n_val = 2.0
w_val, a_val = 0.2, 15.0

# 1. 閉形式解 (10.217, 10.218)
term1 = (1.0 - w_val) * multivariate_normal.pdf([x_n_val], mean=[m_cav], cov=[v_cav + 1.0])
term2 = w_val * multivariate_normal.pdf([x_n_val], mean=[0.0], cov=[a_val])
Z_n_val = term1 + term2
rho_n = 1.0 - term2 / Z_n_val

m_new_theory = m_cav + rho_n * (v_cav / (v_cav + 1.0)) * (x_n_val - m_cav)
v_new_theory = v_cav - rho_n * (v_cav**2 / (v_cav + 1.0)) + rho_n * (1.0 - rho_n) * ((v_cav**2) * (x_n_val - m_cav)**2) / (D * (v_cav + 1.0)**2)

# 2. 数値積分によるモーメント整合
def p_hat(theta):
    f_n = (1.0 - w_val) * np.exp(-0.5 * (x_n_val - theta)**2) / np.sqrt(2 * np.pi) + w_val * np.exp(-0.5 * x_n_val**2 / a_val) / np.sqrt(2 * np.pi * a_val)
    q_cav = np.exp(-0.5 * (theta - m_cav)**2 / v_cav) / np.sqrt(2 * np.pi * v_cav)
    return f_n * q_cav

Z_num, _ = integrate.quad(p_hat, -10, 10)
E_theta_num, _ = integrate.quad(lambda t: t * p_hat(t), -10, 10)
E_theta2_num, _ = integrate.quad(lambda t: t**2 * p_hat(t), -10, 10)

m_new_num = E_theta_num / Z_num
v_new_num = (E_theta2_num / Z_num) - m_new_num**2

np.testing.assert_allclose(m_new_theory, m_new_num, atol=1e-5)
np.testing.assert_allclose(v_new_theory, v_new_num, atol=1e-5)
print(f"Theory m_new: {m_new_theory:.6f}, Num m_new: {m_new_num:.6f}")
print(f"Theory v_new: {v_new_theory:.6f}, Num v_new: {v_new_num:.6f}")
print("Exercise 10.39 verified: Clutter moment matching formulas (10.217-10.218) match numerical integration to 1e-5!")

Theory m_new: 1.217875, Num m_new: 1.217875
Theory v_new: 0.697708, Num v_new: 0.697708
Exercise 10.39 verified: Clutter moment matching formulas (10.217-10.218) match numerical integration to 1e-5!
